# RepairScope-X — Notebook 27
## Post-Adjudication Evidence Freeze, Problem-Level Inference, and Manuscript Figures

**Standalone CPU notebook.** This notebook embeds the verified NB24 original trial ledger and the user-supplied, fully arbitrated NB25 review archive. It does **not** generate model responses and does not require any API, Hugging Face download, or Google Drive mount.

**Editorial aim:** Freeze the available **710-trial, seven-cohort** evidence without mixing distinct experimental protocols. Replace automated task labels with reviewer judgments **only for the 159 actually reviewed trials**; retain the recorded automated labels, explicitly, for the other 551. Separate 680 main-comparison trials from 30 Dream secondary trajectories.

**Open dependency:** Notebook 21's 64 MDLM controlled-denoising self-repair trials are not in this evidence bundle and are **not silently counted as zero**.

**Claims not supported:** general selective-repair superiority, cross-architecture causal comparisons, or fully human-verified correctness for all 710 trials.

### Execution guide
Run **Runtime → Run all** on Google Colab CPU or any Python 3 notebook with NumPy, pandas, SciPy, and matplotlib. Cells perform explicit assertions; failures stop the analysis rather than manufacture substitute data. Results are saved to `RepairScopeX_NB27_Results/` and packaged as `RepairScopeX_NB27_PostAdjudication_results.zip`. The standalone notebook can be rerun without outside files.

Definitions: *trial success* counts each repair attempt; *problem-any success* counts each distinct failed problem once if any seed succeeds for an action; *forward evaluations* are measured with each model's native decoder and are not directly comparable across architectures.

In [ ]:
from pathlib import Path
from collections import Counter
from scipy.stats import beta, binomtest
from statsmodels.stats.multitest import multipletests
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import zipfile, hashlib, base64, zlib, io, json, math, os, textwrap, warnings

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
OUT = Path.cwd() / 'RepairScopeX_NB27_Results'
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / 'figures'; FIG.mkdir(exist_ok=True)
RNG_SEED = 270159
print('Output folder:', OUT.resolve())
print('No model weights or GPU required.')

## Part I — Frozen sources and tamper checks
The source bytes are embedded in the next cell and checked against precomputed SHA-256 digests. The independently reviewed records are linked by the composite key `(cohort_key, trial_id)` and their output fingerprints. A raw trial ID is **not** unique across models.

In [ ]:
EMBEDDED = {"source_trials":{"sha256":"4a9e8ffa10096b214e88226bbe80612b81239a516bfe23eea4a9265956efdc1c","bytes":551401,"file":"table01_all_710_repair_trials.csv","compressed_base64":"..."},"multirubric":{"sha256":"64726df52bd11d0a473d4b435c23ed0dd595bd6d8123983fe4ba3e10a46312d1","bytes":750869,"file":"audit_710_multirubric.csv","compressed_base64":"..."},"reconciled":{"sha256":"d8e9e26f415dad443d8e2f916719f2039a3d3af9fa9a65f6fb5de3127e4a2fa0","bytes":151942,"file":"adjudicated_vs_reported_all_710.csv","compressed_base64":"eNrdne1uGzmvx78f4NxJHiAWydE4NxO4idvm2SQuknT3LNCLP06b7jb2zIhvkjhFv+y2ovz/SSL1Svtp/+fd/q/ru9uLm8Pnw9PL9R/7vy9enu52969/9+Xp8OF+//D6n7e7l93z/uVid/Nyd3i8eDrc7y+e9l+OJvvb6+M//XH9/PXmZv/8fPH568Pu8Xp3+9+vt3c3u7N//ni/+/Tp+JcfD0/XP4o+fddw8fxyLPy//3P7tN89bNLV1c3h6+PL7eGvx6ub3f3dh6fd6wdfbcZLusLLb9+LfQO43Hw7ytjdPX07fNk/Xt/vnl+uE30DvNxcvFV1Ya9psYZ///X649P++fPFr/YX+//7cn84Ghye/r64PP55rezxef/4/PXZETa5wab4sOAGC/Fh0Q0W48OSGyxFhqVLrwD1syYtLF02gk1usCk+LLjBQnxYdIPF+LDkBkuRYbPbCiobV1D5fTTefP9TAza5wSYv2Ho9C26w4NezR9rHw8v1zw3C7uvL4fr2bvfp8fD8cndzfXi8/9uxCdCtCdCvv9s2Abk1AcUb8p/uDx9297Yw9msdMsAfltXRkgNaiokGDmgQEw0d0DAmGjmgkR1NtUw4bhydTpWYNS3WINmhuk0bLOHJrQnSWpsA3JoA1toE6NYE6NcEl5W8ntxgaX39LTitYtakbQKPnS9LYnKDTX6wbfsb3JoA4vc3usFifFhyg6XIsNltvZaN6zWPMyyWxOQGm7xg6/UsuMFCfFh0g0U/2FYTUnZbg2XjGqxafwtPq4p1yAC9DgcKspIDWoqJBg5oEBMNHdDQjuY3awpPq4p1OPSaWyTdbLzOsJg1LdbQZTfLEp7cmiCttQnArQlgrU2Abk2Aa20CcmsCWl8TCE62mDVpm6DeYQ9LeHJrgrTWJgC3JoC1NgG6NQGutQnIrQlofU2Q3VaH2bg6zNS1CZJbEyS/JlDs7FgSwQ0W4sOiGyzGhyU3WFqLJwvP0Yp1yLA77O2Fp2vFOhyAvYaz8HStWEdINHRAwzUNU3IApk59OXgdrg3Ws7Why3Z68DpZG6wHa0PlV0GD1wHaYD0/i9vT6MWP0XuavEhpbT0tOCEbrAdkQ+X3IoPXQdhgPQeLQApepBCdFL1IMTopeZFSXNLstcrKtlVWvfS1weuEarAeUA2VTzEGr+OpwXo6NXQ5whi8TqwG64FVhJ4mL1KKRio8lRr0h1JD072v8PBp0J899eICOxc4cFWNOMLDqEF/FtWrF8nORc178dPzw/jH1cvT7u7xKm8gi86eVManRj/+wm8XqhCVLEQpIhFYiCAiEVqIMCIRWYgoFtHpCY7KuEhkfLOgEJUsRCkiEViIICIRWogwIhFZiCgWUbasGTJ3zZCpNVGyEKWIRGAhgohEaCHCiERkIaIIRFOnFEKzGYo2u6Gp0wihWRD9oNMPAv3z6VSij0SdUozS0qTTT830b7eGfT/T+NSo7n6FJSpZiFJEIrAQQUQitBBhRCKyEFEsouV9P9O4SOS6X2GJShaiFJEILEQQkQgtRBiRiCxEFIsoW9YMmbtmyNSaKFmIkoLokrWCZn08WLSDWDt39c/6eLRox77ayaKdInhBeddeNJuhaLOXKe/ai2YS/QafLe/Pi2ZBWhp1+rF9S5NOKbkrHTY0qnfibONTo5o7CKaoZCFKEYnAQgQRidBChBGJyEJEsYiWduJs4yKR4w6CKSpZiFJEIrAQQUQitBBhRCKyEFEsomxZM2TumiFTa6JkIUoRicBCBBGJ0EKEEYnIQkQRiEp7eYbZDIXzDrMoJOn0pyj6QacfouhHnX4U6Tfsm0s7fIZZm5YmGPQ38GzjU6Oa+xWmqGQhShGJwEIEEYnQQoQRichCRLGIlvb9bOMikeN+hSkqWYhSRCKwEEFEIrQQYUQishBRLKJsWTNk7pohU2uiZCFKEYnAQgQRidBChBGJyEJEEYhK+36G2QxFm91Qad/PMAuiH3T6IYp+1OnHKPpJp58q6n/+c/fw5a3mm8+7+/+MoywXX13BlOGPv/Tb0SjFJStdikwHVjqITIdWOoxMR1Y6ikl3epqgroBFZ9wNKcUlK12KTAdWOlDSzZ3TK2WglQMj9xJZ6SgmXbauSrJkVZKpB12y0qXIdGClAxXd/Ht7pQy0cmDkXiIrHUWimzqZUJguEIly08UfnfSqUz/VoFcN/VSjXjUKVVcf7aRnIXEPGFm2uLWdR3ArmDKsvi/iiUtWuhSZDqx0EJkOrXQYmY6sdBSTrngewa2ARee9W+KJS1a6FJkOrHQQmQ6tdBiZjqx0FJMuW9cqWbJWydSDLlnpUmQ6sNJBZDq00mFkOrLSUSQ61tlF2XSBqMbOtCQo6VlSNBbQs0A0FtSzYDQW0rNQaxbMvwhKitMPbgVThqVdmOB0jScjWTlSDA6wckAMDrRyYAwOsnJQb47ieQS3AhbHyb5oUzuKFc8juBUEpgMrHUSmQysdRqYjKx3FpMvW1UOWrB5yteiYrauHLFk91OYAKwfE4EArByo5mngNWemoTy+xzg3Kpgvaa9wClwQlPUsSs7j1AOhVQz/VqFeNQtWXtccN6VmoXg9sB7Dt+rkVTBmWdzXW8xmWuGSlS5HpwEoHkenQSoeR6chKRzHpym8emBWw6Lz3OjxxyUqXItOBlQ4i06GVDiPTkZWOYtJl61olS9YqmXrQJStdikwHVjpQ0rmtl7N1VZIlq5I+vURWOlLRXdah471uKJouENXYofHeMRRNu6kGvWoQq648blDPgkKW6j5Aehaq2S9wdXVz+Pr4cnv461j37v7uw7GWu8Pj1Wa8pKvNu0yS4wD9ePf0cLIZon/8Gy4cqlqs4t9/vf74tH/+/H6r9eXp7mH39Pf116MfHf/6y9Phw/3+YS6/2IM8+ZGndZGDHzmsixz9yHFd5ORHTmshz34RLhsjXO5BnvzI07rIwY8c1kWOfuS4LnLyI6fY5D+Xl6bY9mslMtq3ZWtLzuTBmeJzggcnxOdED06Mz0kenNSfc3TbSo7mneTYboE5uu0jR/M2Mho2uGHDmrDRDRvXhE1u2LQO7OwW0rItpOUO2MkNO60JG9ywYU3Y6IaNa8ImN2yqge12DzA6bCFHww5y7LZAFe4fR8P2MQIkOEBCdEh0gMTokOQASd0hYeO2a+RWtViFcm3pFoh5EMmvPdLv0B7g1x7wO7QH+rUH/g7tQX7tQetuj+wXT7Mxns4vbDet2yP5tUeq0h4b/5k1+0XObIycXcjRjxx/Bx8gv/agNbaHcLdbrkTWBstra0c/EG54y5V4cjruIYR73nIl4TnRgxPjj1vy4KT+/Zn8bkyT/co0NbxXSX6Xpsl+axqPHPzIYV3k6EeO6yInP3JaC3n2i3DZGOGaXLMkv5vVZL9aXUt7gF97wO/QHujXHlilPapFCvIjpzWOBOEeNVmuZNPYbw0s3KMmy62sgrNZL4MHPayVHj3oMf4YJw9O6trL7378nJIwq1RnfWr19oPrNS5cNAKTCS9FxwMTHkTHQxMeRscjEx7Fxcum0JK5oUWz+tVISSaQZAOp309gwoPoeGjCw+h4ZMKjaHhTmyup3QxS+8XW1A5KahcQBpQwEBEGlTAYEYaUMNQeJm8uUb9B4VufWjVaJHEFJhNeio4HJjyIjocmPIyORyY8iouXTaElc0NL7oqXTHjJgrf5vv/aPX24O5q/7G+NJGAigUAkaCLB6EOOTHgUDa+0GeHYzSC1X1iVNiMcu4AwoISBiDCohMGIMKSEofYwlLPltoRtfWrV6sSWKTCZ8FJ0PDDhQXQ8NOFhdDwy4VFcvGwKLZkbWnJXvGTCS9HxwIQH0fHQhIfR8ciER9HwincoDLsZpPZLsOIdCsMuIAwoYSAiDCphMCIMKWGoB0waLNsWrvWpVbOlE09gMuGl6HhgwoPoeGjCw+h4ZMKjuHjZFFoyN7TkrnjJhJei44EJD6LjoQkPo+ORCY+i4ZW3LWW7GaT2S7DytqVsFxAGlDAQEQaVMBgRhpQw1Abm7MfVxjH/+nts5a2LvoYpy6kfkX7eH+u4fYV93P91fSz/3/3N62+9XT/vHvbNyZOZPK2UHMzksFJyNJPjSsnJTE6rI8/mCJclES5zyee/L0irMJkZkztjm94FMzmE7100M2J4RjIzkjNjrfg0tQ3T2C7Qvi0to3AmA2daEScYOGFFnGjgRDdOewya2txpbMP03Ba3xj0fu4Ypy45rQabuZCZPKyUHMzmslBzN5LhScjKT0+rIsznCZUmEy9HIk5k8rZQczOSwUnI0k+NKyclMTishZ+0aGbYLtCHWqqxdI8N2PZxg4IQVcaKBE1fESQZOas25uz/c3j9sqPTb3j/KXX/8AundIvHiZwUXSnvZD3q/M3043O7v//N4tPhzf/6NVDYyurSRHe21ZEfTmmTZ2GdZ32f5rM+qjdE3X1GQvlmqfmjeod/Spc3XluwXzUS+5tdvs3qZHrhkr+U998AGvNnYv1nfv7ld/wr8ctpSxjjtl450o9FbR523jr28dTR666jz1lHgrWf3AjaybOzJbPoOdn8yiQeOUg8cm3sgGD0QdB4IvTwQjB4IOg+EsdN8CUa/BJ1fwthpvgS1t4LUW6G9t5LNWUnlq9TJVcnmqaRyVOrkp2RzU1J5KXVyUtL6KAldlJp76GDz0EHloUMnDx1sHjqoPHTo5KGDzUMHlYcOnTx00HroIPTQobmHGhe8uvVur+WucbWrW+z2Wusal7q6lW6vha56nStd5lZa5b7L6kJMDK/km5yWnEpKbyL/Vz/jmxTlm72IqSXLWz8zWj9Xbf1TX+AUnpHsP9Jfv+peONIXTTi/sdNE/uxIXzQpyvcd6fNasrz1M+/HSSrKXxzpM4WXf0hh4gbu1++WntH1+t1DwkG9aML5KkTHZp3XMjuoF02K8n0H9byWLG/9zPu2uIryFwf1TOHlb7biXgK8/1rf7VY4ppdNTkvWHdMLWubG9LJJUb7rmF7QkuWtnxmtn6u2/tKYnis8I9l9SZJhs5GO9EWT05KVR/q8ltmRvmhSlO870ue1ZHnrZ0brC25FFz91cUzPFJ4RJ4zTZ2/84JKz/pCZTZWeStywDQWBpl9Hs8yMhWIe1QJNWdcrmdkrghFe/PTTUc41WBBqHe04aAb7ktVE4QZDHQfNSF+y4nD4j3McNMN8yYrDkWv3R3HkT5ef1+6ycjl/302ocYdFs6nSDRxiXtOiRyyasVD8fWJeU9b1Smb2Sq7eK0W/mDFYkG+dEbYDqHxgyWyqdAsfmNW07ANLZiyUCj4wqynreiUzeyVX75WyD0wbLMiXpAAM1/f7jy8bSULDd4NJx3ir7MJUV+00HAfif/3HTtwgPceBODv2sWvajjPxe2/UslZP3OHwnSQtmLy2XFefhB6HdhD5crmuPok+Du2QHcdDtwQgdTuI/X6pltaJQSzq0TEajJZoMPaNBqNjNBgt0WDsGw1Gx2gwWqLB2DcajC7RYNRFg7FTNADHaACWaAB9owE4RgOwRAPoGw3AMRqAJRoIk530q2Nw8XvQ+T073UnPR34OTgb/pq7uTX7eTQbnpq6+TX6uTQbPpq7TPHl4O6mcnTrN8YNfBBgMEWBodGQ3+Pn6YPD1oauvD36+Phh8fejq64OHrw8qXx86+brjct6ymu+7mHdcy1uW8n1X8o4Lecs6vu+m3mVtr1vaV93RL6bflL1eal459UuHde7HUvPKKWE6rGzrrfqpYhKsaV/jG1ZLIZuGOE3vEXoSw7xyapkOq+BJDPPKKWc6rGzrrfqpaBIshictGopS1OwQpzlFQk9imFfOZ9NhFTyJYV45z02HlW29VT//TYLF8KRFQ0FenAPEWSaTzJM45pWz6HRYy57EMa+cXafDyrbeqp91J8Eqe9KyYbVsvGmI0/wpoScxzCtn6emwCp7EMK+cvafDyrbe0mT11cNieNKioSAHUAlRzNEqe5OmigYZg3rEc8/SVNEgk1CPmO29qM0wrIs47XEyY2E+ohcQDlbHK9dQP3tRzcfwunIN9bMa1XzZ3H9Nsh2FfEx/W7KtmQU5T3OaeaZwN0YVDbIj9YgMj2NU0SBrUo+Y7b3YJptSisj0u0VjYZalG9BJvpvG88pVNMjJ1CNyPK9cRYNcTT1itvdimxxOKSLX85aMxbmdLKCHV0Hj4sU5Xn6DcbPdfHvZ3d2//u7u8X/SxQ/DC7HdosHZPfkP08KDMCUDKBkgAgNd6vrhHzs5gzIqKMlASQYxyR7ubm/v9xrvObGU0f00bssHaj6IyvfzNzClvffOTsbGev2vZAAlA3Ri2GTdHDRtJ8vBno7fXiNsSiEoySAumWC+mraTk9WL6lMKQUkGMcnE89WcpYyudjyfVglqPojKJ5yvpu0qfFeFkgGUDODCIOqdsxeN8xMXp+jMs0eH0F3+dOALhT5CT6cVTtEFocbAWv504AuF9kKngz6v8IxYl4DHUQASudBD7lRI5hT1f/jMFwp8odBK6OuLPGZMnSk682zPO1RNfjrwhUIfoYsxdaboglDPUDX56cAXCu2FMmLqbOEZsf5BakYBSORCD7nFmDpT1P/hLl8o8IVCRaHn9wm4XYir3OJT5d6uKRzCFk8FyERDX9Gn8ZZbvCDaGM54KkAmGvqJno7DfIMF4S4BjqsEpNKhp/Sp+MwtviC7bvibitXc4o1F4yAJ2dOl599S1Yh9UxpApBh6Ki4G6+nSy4q9o96UBhAphl6KmWF6rvy86jqRbloHCHVDP92sAD1dusobTJFiECmGuorfZzAB94xiruhpmTpb/+lPB75Q6CN06YxiruiCUMet//SnA18otBdaPqOYLzwj1n3TP6cAJHKhh9zSGcVc0Rmp9UJV6Yxirmg9oYvfxKN6EDIK34OMjR9NjLrXIKPwMUhPLMFTkFH4EmRs/Fxi1L0DGYXPQHphiR+BjOI3IGPzJxKj9gXIKH4A0g9O+PxjFL7+GKs9/hh1bz9G4dMPJcCHWzgq2SqeKr5ZBn6rqGUDLRsEZvtnspGytX+vqGUDLRsEZstaf8t6f8vUlA20bBCU7d2kI2Fzf6tYlQ20bBCLLY3K+S1J915pbD0HpFE5vyXpBqwzm2R+S9JdWBpbzwFpVM5vSboV68yWtf6W9f6WqSkbaNkgKJt0fkvS3ZjvL0Zp2UDLBh3Yfj2yHDa0OKVxyp6WsV+CCKSCQCp0kno24XDKLki1Btzyx4NAKnSSmgVjNTPGaqbaUkEgFTpInQzWnLIzUisGq8nYyynbSurw1o+cuDpT9rRMrWA1+fEgkAqdpC7H1ZmyC1Jdg9Xkx4NAKnSSmgVjNTPGaqbaUkEgFTpILcfVmbIzUisGq3JcnSlbUerZyyXa4lJs5ZafKmd/SimUDULZ0Fn2Wczlli/ItgY0ngwQyobOsrNwbGfm2M7UQjYIZUNH2ZMxmlt+QXblADgZr7nlW8t+/Yo3SdyeKb/wZXBVAuCkDBDKhs6yy3F7pnxBtnsAnJQBQtnQWXYWju3MHNuZWsgGoWzoKJsXt2fK1/laSaFsEMqG2rKXTqS32vvPrfT+c9v8jnCrvf/cSu8/+7KJ7j+30vvPbfM7wq32/nMrvf/sy5a1/pb1/papKRto2SAom/j+cyu9/9w2v//cau8/t9L7z+ZssFHOb9OGiwat54ApiaBlg8Bskvlt2lDOVnEOmJIIWjYIzJa1/pb1/papKRto2SAom3R+mzaUsbWaA6Tz27RhW7YEi2w0MbGlRY3EntHIPehLYEAMAwFh3k9dHBjOnEXuYV0CA2IYCAiTxT6T1T6TqQ0MiGEgGMz59FOC4c471D40n883Ahiww2y+/znN+CvI3mTxlHJq0jYTTsQDch4IySOZWE5N2ma8iXhAzgMhebLcfzpl7Yl4QM4D4Xikk8ypSdsMNhEPyHkgFg/K5x/kzz/YPF6jfP5B/vzTjUcy/yB//kFuvL6Ur2xQPtMgf6ZpoDzLfSLrfSJXH0NZ7hNZ7xN1eaRzCvLnFOwRg6VzCvLnlEo8Z8kik9PIcqlK6WQlecCSB83lnQX55VKV0sZK8oAlD5rLy6yxVzuTrSQPWPKgqbzJYLpcqlIaWEkesORBK3nff3ihGPfOS/n/NAtLHrDkQXN5i3HvvJT/z7Cw5AFLHjSXl1ljLzPGXqZ68oAlD5rKK8a981L+P6nCkgcsedBK3vcktvJ676xUpTTXkjxgyYPm8pbXe2elKqWzluQBSx40l5dZY692hm1JHrDkQVN55fXeWalK6aklecCSBzXlnSULbOau7Molp0o4pTBxpAJbKnSTehYTyyULUq3BpyQA2FKhm9TMHquZOVYz1ZUKbKnQRepkDC2XXJBaLVhNxtNyydZSX5O1eHH1rGTF1FCOVGBLhW5Sy3H1rGTFFFCOVGBLhW5SM3ustshQ5UgFtlToIpUXV89KVkzp5EgFttRa2af/D/vijQI="},"review_A":{"sha256":"df799c38da4dbca19d0c78e8ad1a7049a9cc38fd020aa6e2bfbcebcd280fddaa","bytes":62371,"file":"Reviewer_A_BLINDED_VOTES_completed.csv","compressed_base64":"eNrVfWtvY8eV7ff+FRzDQHfbtF27dj1t+AKZyQwQIJkZ3OQ+gMmgU0+3YEnUkOy2jZv573ftQ6pFSnyKx3bbjh21eOpIXqv2Y+2q2lXSor25fXeT23w6b++v2g9vruq0zN7O5ss337efpsv5VbqW793NZ/m63ciX81Zm8/qmX91+1+Z386vb5bSmJd60XI+cprK8mt1Ol2n+Hb65enGby9i3727S7Rs8/P2bxbtS2mKx/lb78W6OP2HYm/fp+sOTC/wCZSk/C7/G6gO8rbbbgt97tmwvaJquZ/X6huzXX5fZu9tlnf1w+3XBk3me5Lf4moKyX5P/++q5N/2O9d+/u57ldP1h6PTskQcHTHV3jmopnQyZzMobpdi6GnttJnnmqHVwNRvTUqpNObIqWqWUK/iXiQ+vf9OBytvp7/44+/0f/zRpt8v57O6n6fqXYPOlms7fv0lvlKKpGv7+5Pfv7q6vSlq2xYTcdEJqOtFhcnU7eUB48qr917t0vZjgM4A4YfP6m8m7BUb8Nc9+bBVPL5Yt1cmsT/7lD//6uz+++d2//vn//PP//PKTF/pZcM/u2u2b67RYvtH2OaBvjT8MverZKeMqN8MxOG+rzbE4a3LOvnlrkks519Rzjq3UYIMy1dUCyjjH6I9Bv/Wr7CTgnx9wJjf5HAxMvhAK1ph/oSevFkvwUydpMQmv7yn4ZrKcv7stwwcysyfpVr6YDFMff1jAgibXV7cNLPCFLFh1GQsYf5gFZ1WmlICv8UqTLSrk1H0zwWdF3unkok6NavbKF8PsXDLBaJtBXwzpdBbwq+xi4V8ANSCcXy3f3rTlVZmU63R1M1m+TcttVvD2d9fLhRgIm8kreC6wdP0TaIJN3FyBx9vvtoxgxcELcxoFOlxGwaHxB4dNu8mkPDkqLnsfPadeQuu9Ku9VaNF7a52OPTJ8lKXWjVFd25gbaOqGz6LAarXJgfwNS/jf4q+3/E4U5P3ryWcTrSbfTjBs5XdaKm9Bwd275WQG175pCxh+nW6H/zYxlR1cfPLCXkiGt5eR4Y8Mm2rNNcTQarG+JvbBZNM9qca6RxuZQ82Ua3EKdDVFwdjkYAvJZtOsodPJwK/yfDKmkz67vp79ANjzT5M+2NCiXfcvygw/KNWrIa5PUp6BqHeDacTBT5GfLH+4EuI+eKw1NW6Dmu8WN+H7r/Giq9uvrdWHQ/IpDz9+Zoppnqn75GOmFnxIVJPtRhc4GhNNjSBEG2OqLzWyyjZr33NoQTVHzGX1xj3h1m27GUzQNl/iR+PrPy+HgNuv+vLtBPN12Wfzm8kr+3ozlC6uftz61L2e/AD/JJDhqbtWlgkp1gTzp7y7HubPN5NVPAbcMI0vX/h9YHp/Bph7Hn78zNTDf1uNPEZZzoDMkDKWATG7rl3wLdeaAmJmdh1TW2X2uRMcubaA18aDYHJ8Okf/aTZHbrncRGDlrMPKYX8+sZinHCd3sIa6+AbvrIsVhqvE5f9x/O99ycvXGPjli7AHQc9EJyO47+HHz0y5Y7pVTaYgyQgpBhWdgtutimHkhHwvWK2j9k5xKq154y3pHgiJS6EeDiJod1n5nwWZLQBfkfrKvP7MiYkjzVyZq5Xoh58vZq82ccSkvp4tMOX+mP7S/u+ktuurm6sl1MFj1wvTjqdiudvRnj7kCa5d5xgM5qOO4k277pUrO3LNayOJnIlZFReiKsVUxr8o4X/dxKDdXjPfdqKnoau/tPCfB8F9CGUrTAVgeM69wYzUBq6L9+nmbv2fXt6m6y8gJg7O01MH7HpuahiKBemY794nJAKuV5eVD8GZ1pBRp1IGEdMb3CrnpFRJHlk0kjitkEWs3rpvvu41eKRbZXYDDyeaJcLnzb5fCIRmcjODdFwIkN88TM16tYAT/WlykzBh8/WsfH+fK+/I04jOQXP3TD1v2E5kc2NbBNboKacUTemhhOK6SR2Zl3c2WWhAjnCgtXckAdU5ZGnwt6ZFKruR3Z6xvw6++hC+0ZrzZuueAbuem/qScwtMUBm+aM419NwoxYIJ28jaVJE55V5jZW62ORWplZI996RLh9Q7NFv5KZqrDGrxLuP3WCVCOoqIcBKU4BVu7tp8Z1DaikmrisY2lJIPJMR34oNgOj4TzN0Ddj03RZhKPjfHsVnVvUkKsT0n4NYLQ7lBvXW43axq0Lp2r7NNXSe22kBjhHIQTOS7TyXaH27Lh+lZr6Sos5i8CgOkBokTEtBgJsvZ5Lu2RJa5geD9M4AdL94BePxvQPlBoLk3160v6RylPAx4hO/6NdNnvuWwXu4uxYp5DF0QOOpKSEqRECCT6qYW+AirqsJ0hnfQCsrBuk4aMs4G7w1c8j59sPqVD5WL9lUrvh0KFYNqXlUperq6loR/VdQbChb7Api9GPpdBaPnEnBG8QjeAbGvUmbnoCJiIfhjBsQtuVSpGCQdBW4b/4Y3N1pVU2A2HgFRqVT6YRqeXzq6hAw3IhkPRYvLyThaQyJtpR6kSSESIgXp3eqkkPC5gAxP/A4z1Btn5gx35GBCPZEhyzbnYsOpZOypIJ1OxtP63QL+T0rU++MmiPEjEuNHtBJ/zEqSJbJUXSxQNzE6p7VDOA3dI7duiCLBIaKYprQHeQ0pTumkoGugCjvreCox/he0knAOGRzGiBaH3nJw2DRk1xjeqZoK+wAHmPNMvkRH1GNjVwsnxbXGbEmqfQoCHdInK6sg6PVJ0eILosPYm1UK+YVkQfe1bY4rc8DgyavF29m76zrJTR77fLJ69PV+DuLFHDzTII6/6zAfUaloRWw60ZmFi1OeM/LNStw6Ja8gqaptxAjsLqaeHSsuSFJDqBCwZxjEWax8u0XIN9uoX0nNe4Gsa/LvPy3fYuSwkjcZliZu4L8G31Vm+HyQALL2o84hyI0X1t0FUd1NQ7feqJygV13SEqhzNkoB/RhaUhp0kY/RixyrtTttKbualEWYb9TCGUGdTDzir9Tks7XTonujIU8rjjB6y2jWz4JP2m81mqa58vWN5oNsGP/3JbyhVX/nQFHfj5meMeTgo9PSjfNN2RY0/s7W5tqqjYGLJVshK7iGwkBcI7viEGLOIdYI3QuHVcrTCvc//p4BMusvkPfffTXMwenq95myeoryP85n37fb+7l8PZvdTV794ba2H/95Pp/Nv/rLT3dt+Or1ag3otr1v80F3LSZpMiwsb64WbZTJ8cyE1ePC9gutn4E7n4/7kUenGnmQQpYK/+51qTYXJwXZqE2vmOkBKRN1VWPnlDIpZEbNeERr3xWicjM/A+4DwDdt3oa6w5AS3Q7lcZG+t9+ttJqsIDyBfQPrHaUGzSdBbtdB9HTEH4049CSSHzgKH2SVXnlnUkVs1QUeA4A6HxwCcKzG4eNasrdJJ1uzqakVHZSvfBLe9xUz87QI8YcK33zVpWrz/vGKjoZf0RJqkQwJlOu8xxpZ3ZTXT/Q3k/JuuYCK7nAxeKbtBNpM67ylG9KHc1L6+/DY35kV/X3e7tLV/AOSRun7l0wvecfBsdOKJAcijAyHpHrkrhRDKBTrW0kmMtWma7SSkzYNFd2sCwVfQ8sxW/00E/q9/DLDmuMMP2o2/1DCIB2eTv7/JWuX8NPD2peA7OCyXw0rFeH1UARGDI5h8mpw7zrIknK6Fq8DkbDyJsvBfAYS8MDX8mNeIJKMgD+PgD8fwb8lDa1s2GnM/aZMs7F4m71rpTqbyakSu85IfaoPpFurudSQEmJvLV67C/FfL7ut9yZBdi3f3U1mt/A7QzmozjDV1xJst3OHkWzurwD0bgzo7QjQ22PQtwjdi/iaW+XcEiZ7S031bnqCLO7OcoVMIIIUcKZ7DiISYswlF0z9cCH0G9uMwo6ZHwbcb64W94sleM59hgc+x8cJX7vPg/qWjNvr7P3zidjMmC/zRE/fdJgU63NUKSDR9Ckqm5NWsg3MQhtX1ty9RrITQIsnmAVso0MuyE4k1yySIj6BlC05cIwaJ9QMieQrd0+NlKtJ9roM6/sSo1eKmYy6/3TwVA9l1YGerycr3xTG44VH4+WYn/IGKHOFFmPkpClrR8ZqD4UWHLeajEk9qYzUtQRtrW6GK3fE8QoREPM4vHzyZFPSltEAdxUm/zDAP3mVFos2Xy6G7337JHa8/HS6YmVyv7iNJz59KXogjsePGY0fc4QfiK5uVTYlZWV8dZp6cqaWarwLrDtFSbKgL7KKHjqbLCypNlNrhRcsZRR+VvTcvLteXq0MSCKEUERrCja2h4GWLRPZoOav/4FBf11e3UhuICP/iqHyz8svX7A6iR1tD4YGOoWXo+84OHaKIG01FHNjFZMKmTqiuW1dB+SysiuEdNUKSW/uyK5U8ipQRUqsZL+I2bGFdX94eVLjo23NLPuUVitt5l4zI0NdS+ZvJsLpEDSkpLFaXQNBqyU1KcZubwLBT/vkBdMYNOgRaDgydqqdUdlGA0FdgawtCZJCFx9SSQ4RhqtH5lVL6Cb3nnKiVH1x+KKV2Fr4uGnQY9DAI9DAR2jIIdtiKGrSsAK2prbilIpajAPqDxrEwU11qYvDhyGYMOI+uxA8JzZ2ZBperXmQoAE2Xv2AkCLexqy3SFF8/UDMozj+iAIegwIzAgXmCAWAsyA6t4hAnayNnVorPpLTsIkOVkzHeztrH3LUpbZsK34SO/xllUrPpWDf9kkSCjaJAPrDXlbEh8nqSMVivZn1wTTeIvW9bqnKqgTe1K9+nLz823Tyt8nfHnPzNwnmbJ7PzmZh8jKOnr7pMFOsQowO0QDyD8FaG99C9cmTkhDujIXyy95UTrlmz8YZ5Gaq4QGuOdh2VjAfqq67Nhcs57P6rjSpcq8KJfecEK32rOpNYu6kKiJebQj6Lx+xMXCt46q8Iky/3NyPsHoxgrsdgy2p+10S4p++6TBbLpaOHDhaM+xATmQVlKRJIbccQ2VfuNrAXFygaAdHGIyGjamgdDbhLLas2u3gNugC2NNJ3TjEsmYL35X/fyxaaB2Rvj3B17nx+OHR+DkWemQjWEqy5yYkxPrabGYH5YKMK+rMsqUMKYGLUJ3WV5uddV1TywnesiqXf0P8+PH4MaPxczQuWc4xxZ4lI+udXdMmICnmRMb5HApp73PHl6WRjgH6MjHHoGswtJUAfvT8hPH4saPxY4/wYwszU9FZNdUsBGMvShV4N0g4a5BOQ8x4mItXuWmXKkKWAVFIpqEwXW+/IX7iGPysBPsY8ef+TUf4aUjmPFkmY4hhId2oalNpWadWdTOq9wLyXHGuDCdnWmBvdUFiUZw/vzRj9u3sPJLcvW9zWXcZju0Nn9snpLz8y1tZVl9M1jsbv3y59yjlC6PGY0uPxtYxPSoHlJLN2cWBHmR51XTvgs++FjYmaB3IJqR35E2i5mysVDJUaqjutIL/AbZWud0+UbrSQivZ83pLkN6W63eyvbS/Q26+N8vbfheSPJBE45HEo5F0LGVIRivZrR9UK7JY2UCAhgqiiMhDPWiK8k80eJfPrZnQqWeTTOqZIJouJ+lnLx0YPR4xdjRijsUihcTNakuNXCZP2lHOPqTWHCkFk8lW+aSRZEMJdWZdOLYeTbOifeuZudwe6/nkT1uLMiY8TPiN+qZ4PVlC3qRuy9P9eTadLN+27ePJV4uBnpfg57S6glGXFzqPvuPg2Kkl6J1kuVEK2ivk2irnyE2T6aCnkamputSCCQ02hGTOutisgaLVudd6zjra4c1BA8xx8tlEisbDH7yWPVzAcmUq+pGpPGw4Xc4mL//2YCF6qB4YMwYHegQOjoydRgLKpSngqpOcMgo2IeJjINIAitQantBdq55qlbOx5LQc/DJUiirJXMjBsIyv9dYhA/zplV4z8sWKEUCw5uH1vppOmd3cpKGk88hnrRmxYzDCIzDCRxhxKoWmI3eI/pq88SlRp6plB7YJqhavWk82QIsiGfA5VtW45sw+cSZ9kVVsB49VFhZfT75asaDuv/jSPpjFg2NaLJFFp+vZbXtCgRDgxiDAjECAOUKA6iyT3iUDTdmSD8GXkrNuLrkUlQnOAO+YbOPuWg/BJa0VAoZOHLtvl5nE766XUiqL4lii2IX8W87hrveV+g97WvbtTHxh/BhQ2xGgtkegDs32rDUedBAe1vkUG9lCzlMLUlvJCsox9NR9ql4lmEYn7/BnLcth+blQ76ssDxN+OJ0/rAoPfyQ/LEE+CgAiM97drVuJrNLZ6QSTfZjtg8MJzyfhcdn3+cH46ZuOENKt6aUnVhGiAdlrLMpG7VtFbKDkvQ6QiXIKpGeL7JWoeIgSWxtR106dXUDe5YKGoPC4diwRQVR6fIgOK7uQLRVhIzzsFeqDF4rjsaJHY+VYkA7ci0ZUDmRUJlmpt7nWngvEnjcI1B46XYecIcxtzR7yEIQVV0Isrjn9sbNi1Xis8GisHAvUCWlQ6TH1EikbqG9poMOmWqSokOhwXiYln1yBQZXaYmRChOiI6robU/ijZ4XGY8WMxsqx6I1I3JKpMZs+JEscg80xNkQWFSNZSkhq5WAI3FiyvnfpAaO4KwPRhzT3o2dFj8eKHY2VY4EelqA4eRDAcgAcgb5mLjF3GIO2wbuCTKuQHM1vFLzqwULrRd9J44/1o48rlsdg5YwFyBPfdJiVGlJLUNzZMSdlUw5y8ranlE1FXhuj1A3hw5rJphnXoocdGeOQh3lnTxLg2wX6X5oVMx4rejRWjkV7hInqEUFqoiCtaxQpXzzyslyiaTCRAFNqjaFStA+MQENWkwvs2EI88kfPih2PFR6NlWPRXjWdyXpOJjQIwuwd/rYJxlJjkl1iOisdhRdnAyvFYKQivCRnguykHI+VM8slh7lw43FhRuPiWIyXlfdYmlc5NqS/1tYuxzrBRVOwgs5eM5JfxHydpMWYD96WlKypSltO+RewkPCBlJV10MmM+PEYsaMxciy+t8LSWQtA5yz77RHEjUKaxTl3pMaUVMmBQ7QQLFFBnZTExYIq+K8O7f/R+6xR1LwfTc37k9S8kmNY8EwqJ+Uq2xQyzCKV3ir8WCPP2pVUSiWTXIidikKAd8S5yg7WZ+y9v7jIcnydYyiwHOiTa+N4VOnRqDoW9EnZCB8VbLEGaTHSY5UrcrPkdAzOmKjleFHroXE0qthimk+9QeBbr2Nzl1G1+2DRPVGyLPKVkPX0YNE67gh3m0tYam9t0qnx2LGjsXPMvbmCmFq5OkI00QjnqSZOdWiu6xRRC8Fy8SqFgpTAQ90okTWOvId9tfRzsLNVnf9857Evrb4Ka1+n730ebOzqdqhibm+KcKepfXf5qqF7/qKhm4bsAwWOHfFeQ1Mi33W5Gc/G2jgULk2oEeG+VY8UmoqHAVGJ5DhJr+Nz9iLv6Cjwb3LCcdUT9OHwI96z1bphOPx4f9I63f60/wTk/dlHp0cAX18O/uGhU8rKNVOBMNIuFwtUYOwFgd5ZZxXy4KYsGy99nFi5ANGIzFhrp3Suyu7oNHAO+DRWO4f9O7midM3lEagwl1NhDlORYoY67wSNHozVoSMLVggHUI7S9MdnA1oUF+UlrDeXLQgpTtuUVFchXEjFZqt7ejjpuCZkOIBN4nVsvN8WFI8g/8KZZwP/rMq8u7QwD2dkCFE7+pBCQrSGFCSowtKrNtV1S13XpuGglG0l9JiMMxZysFLsnbI5v9a40yqeckEPXKg1F0+DA9Fn2n1LQW4qCO4Lom/x+qPWYUcjyYxF0hFLyamEXHyL1ZQeNUOd5NZarY2rbo5b08XYqm320XIwrTSYD0U5O0yZ9G+QJDcCSWfUIt2lpUg3bZxIOVVDJCWnt6KnJAeKMlsqOYfStaoZqtIV1izHJDQy4ZRkB1cK+vzqyq7gfgpHm8vAP8zT3d1wZ8e6vzD+o8r3DcH/5V//4wkx//kSPs6Pxowei5mjMb/KBiBCXGFpYKOQV4En57K2SkudPmSNhABisppUO2mHNCAHlUjSZfeLMbNzD7eJ96Hokd2s+5Xiu/8NWsJotPBYtPARWlKooXZTgouBtIkkpx6YwAVkYmjeOS/b54q0eIpKqUpaBQUaLZ4r8dc2mL2mcm8pcTRKzFiUHAk0xCZhxkeStr1ce5KWvZE5E1Ll5jRTz13LARQOiEVMRrZ5ZaWp9xJ/QUs5l5J77+XVaJzYsTg51oHOUpKzCzE5o11TxjZp6tu81FS4qRJajiYrEzPsBVKSpB1FzbVa9lXrj58TGoETP1bW7E/Jmh0S4hAgzEEDN0/ddd8y41uFvJyI1EjWpIc4/BXEvg3BuAJRDznvi7N0/lbskzjZL2DWwUR7tRFMtuTMS/kZB06ceD0aS2Yslo4JTANdya0ysqveXGToGm65SpPGUhUSaJZNX9qRl8bZ3Vhv2QZp99i6a3UElmi6T/CDAac+fOnpaUedjY+PS0/Po9Fjx6LniGNTVbnGHbI/GRsVyMiuOR+DRtqsLThybGJonWLTBlKmRUibHCVbCI3MKEZ0TkHmaQ9UDC2yUU36ze8/1PByIs0N730j1I3fqBNsXk/i4ClOLMqcOezx49MQdOmmVWPJdx266qYPRS4oSiXi0Uu2ZVXUqbTYXUHW7HpUhL9cVHl9Ccr+kotValfLtdndYpL6sg2t04Zd7z8k+dPbNvxfmQGlV58iL3/9UH6sw3Pz1udX37Xhx+ARL49Ivjx7327xZ/y81/cN/z8dfvoLb3fD7In9iXvkzxz2+PGpTxHhOnmLoNC7lstllBM9wg5wIzz4PnzDe7nErsAtdbnjS/k4nNMux2A2h+/teFDhy9kSk/KHdvXdW1ksxESXA4hudenMvN3gNx7IWD3ghjAhTxg7mcldVYt9qyHe7QM5xhNrvWcOe/z4NHZpCZuAHBQ2ElUrvQi4VNtULMnJASi4E1Xg+OUMW/UJbkVli8BdYk9HQSaz6yTUUEe/7/ck/Unb9dAv/C79hOlIvD6K8ykp83o6md1cLZerj69ulw/P2fvHjCw4/TCfPXQcx8DJexGAcmTN+2fCbJ4Hs3kKcwrRy4bDqggeg0op2VZrUvQcyIfkfPOKIayNgVSQddtYguyzYu5yJPeZMAO4xQQ4bQF3B1+Rr75v04f7lPBJAJiC/Ar1zVzmU7z8kfuWkzcrvN+n63dSYh9wDs/E2T4PZ/sUZ8BbA2atzi6Exio7y4ptS6ol05k80s7cdM4eX3dC6Cwd/huOhg0CqHoWzv/04Cq25+42iuHDJ2brRN/DYbKXq2Y0ktfHs6Dcn8s/a/ATWG1RKkjDYw2h1GpUSVVfAaaHW0aq6BDvdIbrZWlBKodjak2ykk1yUZWjvbA+SjLGB/e7NM/XAu26leKnUmNQcsI4qIsg5ksg5qcQt8KqRSPtWpG5deTUUEKqQQtlb2MIyjRDZLMhZzWT6UVrDlk6jUAwJf8siA8EPXGvnwvkrwdUwwe3MUS9lbe4B1wE1BAk9wW6QBdhbS7Beoc3zj25TgmzFE4EM9aEbgFq695IA1cdarHQNPDTIemeVbPNdGQbBfPdUOhjY30P9XGk9wK8oTCf3DJl/AZO+lA28Yyhu4ZMc9ItDweIWkea1iN5KwI/RivdhzurVErqJvvgvBdHLR2ospXWLBxSWt9ptd8Nq/DklkCafrLuSiyfThapfC/ttifvbudXd22CN9x+1+4TaKRmNf2EWa4N8B1utJJR08nN1a307/7pbjZ5qVfDXx7Sj/KLfPIi8AXw8/Ph593wFznNKz08Y89ChC3Z1Gp1A+QN8Y5BT4MOhGSRhlLsYpXLG1WS+zC77c+Bf8/U3sZ3xco3O0AURdhusqzLySJY/eLDTT7Qf6IMHzTgi2AuQNs+H227b7IHztFXSHCBGiEwOvju4jKRQoZnNaWuXHSdkfeRxUe2E77BISqd6GdD+2Kc7dk47z+6+OwX7MRc0DasoqkaLrnAa8s2uJxdQWKCD4zKBgkIrCBbS7CCYb+2UfA3DeiXvZhvr+X+yvPcXYy/vhT/PQ4+Id2zLhdVhg6pPRrMbI5k5eaW5BWFiOndTfI6lBwhdVrtyvfIrdQS4seBP9KWtJF+r4pP94QAf38x/nwp/ns8PIRLIp16ljVAl5V0CmhNKCglmNqhz3PMxZuKfKdZRVa7LGu1WcNOWvxtzP9wMf7mUvzNbvzJeKSQSTvdunHBmeyc68XElE3LAV+onjK1lpBDws17ROIWk2PvIhyQ/23M/3gx/vZS/PfEXC2XEVLp3hNDa3qtOiZ2rDEmDp5SkeYNvVUXQs0VCX1XtgRZ54OH4sq/CfyjugB/f2n89YfiL+COSW6cSpGTZDG9O2LFIWRdKDXfmYkTU7EuVF2bT8EaWY4lJJktnIC/t3vS/H34D5h/hbR+xcSQ32/SMcipU/qVyE99Eeli8PWl4O8JvlU1ip7geIoNKkkMlkKMVQVeqOmka4myMacnsRJtQklGKShdUtpGfQH4Z03+B6jv5rNFe7ib8xqfS8lxS8fepPn3shc66otx50tx3xN0kU7aQjnlDhgJuMLrKBuQ3cs1bYizTLp4skhBXc3CS/ZJSVNLBY+k6Dcw6fli8O2l4O/x+L45ucAd4lUuNNVy9ALA92CDSdxrS8Zajkr2PMEUmrMIvylLn7DIOga+ZNJ/csas325PlY43qMLYoUVVPKRqQ/Annrx4xtBdQ6bAuFK2cjQf+bor2ZbsY63a6hI6lK6Xw+Ml2eq7kf1OmatRshRtdIg+HlO1T4tjeya5vb8cx5jJjdyblmfL5XXDI3cnz278sBfRXoCvfj6+u4dMC/L14BEYOzL3zBBQqcqOiyyXr0S2kT1Tj65VS3AxIKM1ZVKygZHTGzofX3Uc3y1ol0M+DnS/GTz2YlfdMboLQOXng8q7QeWKfC9iwqpWSZR+7dJxhZyX7fnaaFMQF0uwtQwTNcoKUAoOObpSzdefBdTHk3YzMsJDzdv7q9Xqzwd8ZcIe2E4U/dmg79+n+uwX7CSgO0ty11NWhVpfLRMpq4K30bOXanBTcBwuQUDFJq03ndwqnomIQ/fpBC9t1dTsvHegfCBi8S7j9yrLxWox3qrBRw/Ogj4Qs4nvBl33C8lbO1HEfYQLQD+al5z6gp2gW8+mIBJ2pcRr+5BbI+Nsl9aOcNk1O5thEUgLm2JvlWYNX94c5j4G6tNC4+hz/8T5vvbeh7RodHywcHsgOh4fumvINNbuXIXrLq4j7fZK9ZKYrXM9IPsjqXgViP7CSTNXjZRcd9mf3WVzaT3qvX0wB9OQxez6PaD+UW73lgwkyGnQ/4E/r79cfRdv2QT8oWndo/1WQPzhfPXqdrMfv5bR0lCWlLoAev186HcPmZItiJw2JOq9yp3Scm0ZnLzcwhvk+EG0yclBLDgiuQAZvp2rLEhnDwlq3TOg38xMFrId993dA8ziVRbL9TW7e/IPeSeApLOBPMOFnPqCnaBKKT3BJdRk4KuDdnKbYpMVUu9ztJUs19qVDj5ScTrKRXAdwbaVwAL4aS7kCLQbPmR7Ek/eXy2u8LqT87013veT7/C1xbTlesvstl/Nbz7MQ/uBtcNXHR59ycHBUzkkoIGl73LhW4rVl+KtjnKjkuvG5hJ96iAp656M3EIWM5wlwqmc6sALdm/ovLqtDQzI1cX385vNkZ2cq33pq8uM1zs57bpnOJvHAuf+LqW/vG3z4VrjPfs4t/lhI4qHPkznywjSYxCkjxAUiIkrZKVuWvcmG9Ots1Vur9BIL52irFPVnRu80tD0T/oCIVu3qgfl6Ncg6GrxuNU+3v3yftdRulrtuWvzq1kd6DCj0MFj0HFk8LTkyCpoYg8v1ZDyaM1Wu2i8Jzcc3IycXDaqGaQ/UvZyGewQRBapntqvRMf2dqTBVobd0A8mYUfhwIzBgTnCgVXRh+Iz4jA37SoyIpWLMZBeFvm9az4qbR3BGJhd0DXlYrNzyDoj5aRH5uDVmoTXu1nYrWAFcjcK5HYMyO0RyH01so8/E7JLw8HnKl2uOplqbbYe4ipT4VAiprzJjGBeTc/dGUcN/6Hl1woTx6pgvLp2j5S/gIvttaOLAvfTVx3mpbPcXgBjiEk6WAJ9ncgzVaNzzS0UEGaKVlX2KbjQdVa9tT70XrQ5pxN42VqZ4mMXfxxh54Qcis1ASBiRED0eIcfCdWyIzhm20mTzDRMUmCyCWN9lz0JuCAgIBY6S8tUrIt+DK8bG0irBguLHS0gckRAej5BjATt1Kzeyp9zl7lAxlqyr9jkgjtdunK4tyCklmElO7FkVE12UTgDs5ErEj5YQUiMSYsYj5Fj0jh0xGdKtux6akRbIrmP+Z4HfAf8Kx8QxdmjC6vG5LwjzXsmF4ZFN+3hdFtGIhNjxCDkW22UFxXlpGMpZDumppm2ppjbTHD6BJLTOR+9MLDbAD/TsrMaHSlrzFvURW4gehRA/XlD3JwX1UHTuqRsNz9SLXEQUg9wlaQ21KHsMje0I4NF275SSwkn20vPdZ1day/0sQvwvSQiPSIgej5BjQd2wknvvcieqLWPKa7Iudw5JF8vFltJ6J9ZIvGI2LOR4jecQZJRyagRCfrVSCZkRKePxKDsW9pU0XSi+m2pKMpyrj7L0QCHlFLOGJZGYj1PS4bpCmEv/vmZIiifSBeDjtSE7IiFmPEKOhf3cbc3awmJyjqkRMl7jskoRfzF3bT3JsnNwuSH7QoYcssXjsbgqeys+YqfmRiTEjkfIsbBfWvQBIRzSHSwU6BY2kWQlQ64yti7JRe62Bd2cXOvevbHGpJqTrZRarB8vIadpeR1GKL8ff8nBwVMjlfTOXMgiWsgB84yvoVK89klBKSIXIEAeENyDtPiLtZWalfKhVc759LqK1epwUJGGu+TxL63u4UfYUysCMHozrGys6MWNcaslPPu1PL6KHmEUKswYVJgjVGhjXI1QgdpDlHg2lBjWEFOOlRNYisEF5ZKFjm8QlUUaITeloWacwsjxqFhM4mfkPwMPr74VClY9E/A99fn9t5GRTKXJwmLo5NPT1fX9CfSBqceL3RuMxAsYeVah69RXHWbHy8WpBFVI1FjXGm0OgyWY1JwEiCirU03u3olI1jyoStGn4mziavY2qt4rUkY1l0eLIfLul0NqtmMxRKsRCdLjEaSPEJRMsNW55Bux79Ej2ku7d6TBhWqBbSnpPNCCnNAhX6HuNeym9txjNz7yb4cgGpEgHo+gI6+YBl0Us7XRKehFGwT71uSCKtJZbtYxw7YFcrZ18NW6L8F42V5McHre/3YI0iMSZMYj6FgAGhpzODltr1tvJnVYjjIxKt1hVakFa6kZ3bKNNiAp88bZ6F3vOUkFs/52COIRCbLjEWSPEORsbbAGFVxqXdZdfIxVO6JSZZ8bG3ZW7kUyNZYackNG0aPKmYrVGn7xt0OQGYUgP54F+ZMsKFRlqmu+D7cidhiQlSuTmg6lIkPT0VBwtXXTk66d8ajB356YW04ph3MlzU6C/jKXzkJxaOvnRc0o6WTBzOsdW9JN675N1n0/rTR5//i6ErmLfaDwUeOFL0HPaSUAphEEzvGXHBw89dZEZAOOdcrJWfwbUlJlDiznGJFZNw4diUCLZJ2uxDElOU7XqCe8o5+eVZsdByieXALDg74cOmXJ5lyjJu/k9p50jfz57by1D/f3zG5L28yjb9I1GLn5sIXCqK+Nerm+00I4caNwosfgRB/hJLFsWDTVOKOYwIspPltpSV6VkvPtyWYN4wm1KI0EwcVqOTuLVFrZxvW5nDzV/CxMaLUS/msPptf+y6iTNL9Rg8fyo8DPY8B/ZPBUTgN0I9dYpCLtLTMhuveqwIjqxsFaNIWEVCDH0FpRBJ68MKCaCrm2S+BXZ8G/sYXotly/q7AUeUr+a55s8Np8lRjWoDZ1GIUWMwYt5ggtFCmUrkiBAUMQ9r5W66i7ENkoLwcMdEpSQZY1S2Uy7EZn5AIY2GK3v4CnklDRUnmLl94heqx81N4zMqTjKOjbMdC3R9DP8CxaZznQ7hv32oOvvilIj6xts3K0OpQYsnR39yq2XOT2Q2nanzm1Ej5Cn8TqAviftdp16qsOU6Hh8DUXP3ToIS42FNtqVpGk7aUFB906RyHkzqlqS7FCpUTfinixkwxh+4DHs8zhcefqxfok7+5rEZ4QNPmrtLGWCxLH44jH4+hYDMkVCRRsxVBXlignts5ocNCzgwAONXiXTYdAaTFUstlyj9CMJSP/VTqOwdEn55G0uR1v2X5cHmi++7fHbK026fFpwn6Muv4lZf0wRezWJvcQLRXuTUm7t9BL5+gr1CN8Wg2qaoh9xHs5sZZjs12OEyujVT1jtyTt8GV/uF3OZ/Vd2XEr5SvZNQRy1iXlutGN3KyPqZ3k4Gigg8egQ49Ahz5MB9UQJLTEVqJcltvFDLg7xJLuEXdcZ61UahrSMHONrqZSEV8ikrQ89DW7hI7NJS4jNxuu1riG/obDN1j6731Bcd2z+gwCzBgE8AgEHB47Fd9TjHYxWXKAGfltqDl1tpjuSivXYCmsKRaragvKIw/2KWSVO3mXLiRguJ91dbjy3g6IYAUsljB5NRua1a7nv9Ay2AENTPHgyoSpMPmHb7e52UWIHYMQMwIh5jAhpbpSXM0VTirjezFL0z7rfNBdDMUwgkZQrrgkpUZZIm7IxrTR2og8v4AQ9RyL2FvGov1FLHZjkGFHIMMeJkPq8XKgJLWcKhuktFo76RjkPfQ2B2lXD1cFDeKdYWkEWiJ8ni9Q7wEO6iIyNi6k0DxcSGFW4VsIkUSYhzPLq9v0Xp9/Xcgqy/LPp+JZa46XLzmGqfdFOzmF6HOyvifTvNc66dAsXtschwTTCKp3YyviqoKBBEj70LzcUXX2tshdlvJ8dq4W9/etfkizHo7afvqIpE+/XOVXYTya9Gg0HQnulmyB28q1+Eop9A7aVOqBdcv4S9tC1rfmqtzoooounZROOccs7Roof3Q0Pb53ejsZpr+tuYrjccWjcXUsDwgt6hKr041qg2BRziZlY4cOS9TBVYDyl0YOCFAc2Kbk5YJqGyFiEHo+YpN6JFnuaTJqPJrMaDQdyQ4C8mMEIfzBkYqwpNCMcxo2Jpfvgi5PSOYQlRCDevJRM+fiqfoki5U1ffQmte3/7t2fofG4sqNxdSR5yFHraE1L8GtNFWN7zd73lmyKXSvy2mvbEa904OZ0thiQXW7M0EClh98qV3oMrvxoGYU/JaOw0Jg2dTBlHCROTkQphWAKLKuzSSbUwvjUU3LJ+cK1Vl9b8qRaN82cW7T59XJvw+PRo0ej51iZAJKf4d0QgqLtLRIsqSeWmxWz0whBPlTXg8EnvRkbnOSEIAbMONuK+7XpuVpsrlKuAhJJKFpZjBmPEh6NkiMJQyYwUkkz1WTgymi4tcakHAB4tz30qL3pmlpjOLumrWZfQAtYtM3RxZRsXzk+XDjxau3dXk++mrwa/BsZocd8FR9c3EklHJBixyPFjEbKkfSAU83QPsHBiQWrCv5ftvCVGJI1ZA3SNtdzYCK5/NJxY0ISR8HI1aTG1BFI+RmLamajbLB5gYe1+sxzYs8b/XjUlH1HSJArKzNsAEgWVy2HXKNzITadbLC6F7loSFkfSsgke166Saw9+Nm6NGQvyO6pM/qX+7sNZ/eLv79P76/q5Ie0uG+aHaTJ6nVaitOZvFrvQIYfchvfhzWsY/w22kHmv98NNiKeOS82PG/041FT36hU2ScEjS87WFPRrXuWdrfaeMToopJip0ttNiIqa9U5Q7s0drkmF08Dmx4tQcplbfMlfolh+9B9W2bpUv7hpiwAnpaYuu1uoievXmK6l/nglD6Ta0Ju23tkS3NZG5Z74+7vIpvk9B24mj/eQGTCZbDzRbA/HTWVhnwlBOuTt66WVEuWxXYNBSLHhayX/sK+ZHxga9fOBDJFFVXhQV1T7jcC+4ZmP9yJTx8sCz9n7M4GfkhwVLay5lFIWiVW76AjNNXAyhhwwdHLygjpIvf02RIVlKG3qSRdlNtu6Lxrqf1Xbd1HVp2P9353/vw37MQeUbERAqbcplohqLvRyXCNQDdouCCYQuegoNk0aZMD3Dplo00larCQvhf7n7d54r1cWx8rMebQ/cNk6XIG+GIG9rSvJKWkBQq7RMrB7+fgew3EjmLPctGLNU5RtBZOqBiFwECIt03pEkwoj+5y+RkYeAz9fQ+n1eK5PPLl8MyXE6lErbrm3qX59y+HFayn20atvpwMczEZe1wRIfFykMDSRMgWly1lm5JyibuWxoC5NPbGuyoXgKZUSiw9R2cyySF5l39BcwDQSdKfOTLYSfoh/XRoD5bl6U29vqFw7HAolE2kDy5bGruvB07PHnfkIG6tcsFfSEXubXG5IAB4ctUgv3QEaJWV4kOMSnEjts0bJJkxGGlzKdP+cfr+p9//8U8TZ75Yzr5vt4faOP3b/Oq7Vd/Pd0vZv3ZztZAtbd9MVj0jMbXXH8zkmtbaOiBcTP79p+VbhOH3aX6VMP8XHxZxZZavY+79fuqNvSgIv6sikpBgTiHBqOeRsHvcwQFTzPSkKTNDJsHHF0fSRs6XIlsQbFfWQNw2xFqvdCXZbchNqniFqFIt7TQSSF9AgpR2Fj/dLtOPX+S5vPWeif7FYjkfdq2v3MzOrev40cDdPhd3fibufAx37sghQ6q9M5mhXQMEatDdWAgoVq7Wiky+a6NtU6mbroeOKNJ1Mesdp9A/TtzdWbgjBb6WYvO5831r3BHcTbO2y4U5VAp725ORAkFXmPcp1xJr93DqOfrIxuFBVSgHI9cARB0zHcF99ZvsxP2TvcBvIKfj5POJ/PPh5Llx68qBng54H+jjZ/1z0eZnon1slucO7ZTJ1uZsQzYTgsmmsg/aVt9qLi5FE3xQcoeIidWCEtdiCknqbMF81GiHU9AOz3Pl4UxPHqY6ZI+wqLqUKIsKLbBzISQkLakX033ubCSbVHJTUWBDLkbC7IZzQYpDJzqU5/iTMqttw5tM2nwOBwEf8h+rqPn63pEgsi7f3rTlVXniTcSZxGcCzs8D/NgacvLOtx5jqyalFLtzKsK9hJyQzasOMW7wP0Ym0yxkLcckK17dwK9Ua+KvAfirdcrynych/v8BHK9u3A=="},"review_B":{"sha256":"e9f5e3d371b124d6556b40944520278df900ce104e436e271073ca54fc59b5cf","bytes":61400,"file":"Reviewer_B_BLINDED_VOTES_completed.csv","compressed_base64":"eNrNfWuPW0eS5Xf9Cq5hQJJNdWdk5FOGP/ROzwADdM8Opme2F+ge2PmUal2vKbLs1m7vf98Tl2QVyeKzeHswli2XWMxb1DkZjxOZGVnSrP1w+3iT28P0of181X754apOy93nu4f5Dz+1L9P5w1W6ltfuH+7ydbuRLx9auXuoP/Sr20/t4f7h6nY+rWmOJ82XI6epzK/ubqfz9PAJLy4e3B5k7OfHm3T7A9780w+zx1LabLZ8qf3l/gF/wrAffk7XT++c4QOUufwsfIzFN/C02m4LPvfdvL2habq+q9c3ZD9+LHePt/N698vtx4J35ockn+IjBWU/kv/r4n0/9HvWf/10fZfT9dPQ6dkjDw6Y6u4c1VI6GTKZlTdKsXU19tpM8sxR6+BqNqalVJtyZFW0SilX8JuJz4//oQOVz9Pf/O7ut7/7/aTdzh/u7r9Mlx+Cza/U9OHnH/IPSumpGn79/ROOE3LfkvpWhw/kPpD6oMOkPt5fX5U0b7NJur6eXN3eP84nC/rxym2dtP94TNezifpucns3WWA/GbCfXF/dtl+90a9C++6+3f5wnWbzH7R9DeYb4w8jr3p2yrjKzXAMzttqcyzOmpyzb96a5FLONfWcYys12KBMdbWAMc4x+mPIb3yUXfh/9Q/pCgjO7yYPLZXPk4UNTNhM3g18CA3fTz7o6aRcp6ub2SS8/25yncpPs8k//OM//eZ3P/zmn/7wx7//lwXeX73hCwG36jLAMf4w4M6qTCkBSuOVJltUyKn7ZoLPirzTyUWdGtXslS+G2blkgtE2g6kY0umA46PsBPwfb+GO4JPmE3y+8ni9+HyTJ7Qb3MbjMOdBigAPvwE+9kzxr96Y0yDX4TLID40/OGzaTSblyVFx2fvoOfUSWu9Vea9Ci95b63TskeF9LLVujOraxtxASzd8FuRWq3XMCb8Ec3jg+VW/AqiDU548u+/Ju/gt+fffaIV5jtGTxxkCxSROJ5gtE/z5Dr57OsnwPOVxPpvc9T7Jrd89tA0DABH2QiK8vYwIf2TYVGuuIYZWi/U1sQ8mm+5JNdY92sgcaqZci1OgqikKxiaHeZ9sNs0aOp0IfJTdRPzzw52Ew9lkZQP7iZhO5p/vHj99nnS4+Hb9ZeWAnvw/GHictTqZ/3JV2l7zcGusfJrdhJ8+zh/S1e1Ha/XhGHvKm7ffM8XsztR98jFTCz4kqsl2owv8iYmmRnChjTHVlxpZZZu17zm0oJoj5rJ44p746Ta9ySMm5sMcPxpf//HhDpM23c6Qt0ze2cnPs5Und++HSAl01tzNZPYZLIqDubqd3YOH/T79jd8HoPdnALjnzdvvmXq4ZquRjCjLGTAZUsYyYGXXtQu+5VpTQOTLrmMmq8w+d4KP1haQ2ngQQI4vp+T/HPzBbN7uP+QvH+T/mFlX8883bX5VJrPHmxtxB/DOQbITi7nJ8TuZeLPJ79K/tv81yXd/wSSEQ7hJ8wks7jMYmH9Ot6vZuOEl3oQ9aHomOhnNfW/efs+UO6Zb1WQK0oaQYlDRKXjbqhj2TUjggtU6au8Up9KaN96S7oGQihTq4SCadoeB/93CrMVa726QrQEl/Sv7jRObRtI4TMV61Xt7kJRYXkKqNxi8Wk3BHbAtZ2I8FbvdPvX0IS9w7DrHYDAXdRTH2XWvXNmRa14bScVMzKq4EFUppjJ+o4R/u4lBu71mvekv1b65uTYdBcIPkxVik5urawDc04NMuu/wx9kQu/YhOMEEbbf1V29IrSE5+znd3C//suVzuv6A/P/gTDx1wK73TQ1DZCCv8t37hAjvenVZ+RCcaQ1ZcCpl0B29wXFyTkqV5JH5IhvTCunB4qn7ZuRe837M+ByDyptQFAiNICgOcOkzgeDkPx7bTN5yfCYSnQPg7ul43rCdYObGtgiS0VNOKZrSQwnFdZM6sijvbLJQahzhIWvvCOrVOWRccKimRSq7wdycludCCkcIl3gH+BCDbhDtMUMhLerx6QlY9SFYozXnzcs9A3a9b+pLzi0wQRj4ojnX0HOjFAumZiNrU0UClHuNlbnZ5lSkVkr23JMuHULs0Lzkg45yGZbFVQqO2kmMWYaYPw/R5f/y/9uILOvQfZwwgOODwDk+E7jdA3a9b4rwknxujmOzqnuTFOJzTsCoF4awgrjqcJ9Z1aB17V5nm7pObLWBJAjlIHBIUV8qqH+4Tr8sIy5wm7wLOn4w9v2vg0zAOPly1a7rbPLLRkYU1zIiPPT9erq4ayI+iSr3w3XrczpHzQ4DtkBePmb6yqcc1rTdpVgxcZHPB466EjJKRHOkRN3UAl9gVVWYv/ACWiHjt66ThvSywXsDb7svr1985APFG2jZhZya3V0/Dp4gzeft5n7+rGbfLUs0z0L2/VH3+tUbshfjv6uI81oWzijowCcgtlXK7Bx0QCwE58vAuSWXKhWDNKLAR+N3uG6jVTUFBuQR8JRKpR/m4oxyzv3DXX1EisVm0h/ubsAIFC1ElXDy6epn+JYVR++fXfNOLtyIXDwXGy7n4mith7SVuo0mhaiHDKN3q5NCBucCUjZxQMyQX5yZM/ySgxn1RIYs25yLDadysafS81zoaVtFzqG2eQcKxCIGAxkKPE/1hZur+mEGVzjkyL9ciQ7e1hHkR2TFj2gh/piFJEtkqbpYoE1idE5rhwAaukem3BBLgkNcMU1pD+YacpnSSUGVQN911vFUVvxuC3mqQDwx8cvnq/J5Fx+HQ0Q4hwAOY4SIQ085OGwasmsMb1RNhUEAd0xyJl+iI+qxsauFk+JaY7YkZTkFeQ3xkpVVkOP6pBDxgWiHR/rNs3xpDw93Dx8nZpEtfpB85/vJB46L0LAMz3jKKREiXgz/K+f/8WcdpiIqFa0oRScisXBxynNGQlmJW6fkFdRRtY0YgdzF1LNjxQVZaAgV6vOM+b+TkFXMXi8HmQ9kPqzYmLwTNjAWgQFycTZ4IOj52iZ98Eg//vjjVqEDsh6Y3G6VRN9odQ5Jbrwo7i4I4m4auvVG5QT56ZKWuJyzUQoMxNCS0qCMfIxepFat3WlL2dWkLKJ6oxbOiOFk4qHA8Sz9PyJ4fyMui6SS4kkKd3PIKsQL2FI8Llk1TXPl6xvNB0kw/q9zpBBW/ZUDRb0aMz1jyMG3Tks3zjdlW9D4la3NtVUbAxdLtkJGcA2FAbRGDsUhxJxDrBFSFm6qlJdF6P/+Wwa2rIfa3a/z9V35abr4PFNWL8H9u7tbqZnOJvnx06cvi2ld7m5llfEWnxRJ0KrAvAq8IAx/h7vlkiOCwoslBNG8vH8RUr8CeT4f+SNvnWokPArZKPy616XaXJyUUaM2vWKKB+RG1FWNnVPKpJACNeMRmX1XiMDNXIr8U2oq6SewXC9svQBz8m7Bzz9/AQlwMeXhCrLi7vb6y7Z+2ICaT4LaLoPm6UhvjTj0TiQ48Aw+yMK48s6kiliqC1wEgHQ+OATcWI3Dt2vJ3iadbM2mplZ0UL7ySTivKl7m0CLLUpw93kz05NuJNvIb0v/vJ9a8X65zDYu+ixm/ttI1f3i8Xay778w+v3qjzbQ+tHRD+nDeSX8d3vZXZkV/fWj36erhCUmDT7x8yPSSZxwcO61IaiCyyHBIqkfuSjGUQLG+lWQiU226Rit5Z9OQys26UPA1tBqz1S8zn9/Kh5EJe303uIWnYgXp8HLS//Z5C8MSZVpUwSG51io/pMJQ2J0kfK3DeuC9SddS6mh1wQG++1F+0htEjxEo4BEo4CMUtKQhhw07jenflGk2Fm+zd61UZzM5VWLXGSlP9YF0azWXGlJCvK3Fa3chBf90t+5e4F2WRaGlQq6Td+JWEFDnwwvyvMm8/QWe5e76+k5KTfkL5MF89h6guzFAtyOAbo+B3iJULcJqbpVzS5jpLTXVu+kJorc7yxWagAh5vzPdcxBFEGMuuWDehwtB/+rfpHwZFouziwmffpJ9PFtqYDZ5576hMPlvyGqMlCiM+7D4kz6800T71zOxniRf5odePukwK9bnqFJAXulTVDYnrWTflYX6ray5e40kJ4AXT7AImEWHQpC9P65ZJEN8AisbCmAnNwM1bp2aTV9kZJFJqtCDJ7q9u/0gi3nXV+l2PdpO3g60fJR3vYVlhPH44NH4OOaavAG6XCG7GDloytqRsdpDjAXHrSZjUk8qI1UtQVurm+HKHdG7ItePeRw+dm8Jmgx28f0QGgZFJhaxTEE3GFmsPL/9erqwk8lqXQDv//qtWEocjxkzGjPmCDNQVd2qbErKyvjqNPXkTC3VeBdYd4qSVEFJZBU9xDRZ2E5tptYKx1fKOMy8LF64yTeTdwjU307AxsDOonqxtJVDNQtWJxGh7UHHT6dQcPQZB8dOEXythvptrGJSIVNHlLat64A0VbZnkK5aIZ/NHYmTSl4Fqsh2lWzcMDs2hO4PHi9KdPQSdQAev9Xxg5YVHvw2BIv3i1R14ZDWShCDb4JFlM9tpx6GYRghg8YgQ49AxpGxU+2MyjYaaOUKfG1J0Ay6+JBKcggiXD3yqlpCN7n3lBOl6ovDF63E1sKFZKzqRWuRe5OO5fyXhZ1nr7RNyC4G9BgM8AgM8BEGcsi2GIqaNMyAramtOKWiFuuAsoO+cHBJXera8FcIGYyozi4Ez4mNvZCBJ+Esrl4qp4L/u4GA99sMPOuGPVW5Ffg8BvhmBPDNEfABZEH0bRGBOFkbO7VWfCSnYQgdfJiO53bWPuSoS23ZVvwkdvjHKpVeC/5STP/djjUcWaZ/t1isHxgg8916YiTB/PF+vhAO1y1VEdpvf5xOfpy83bV3dEWJeT0l60XFy4h5+aTD9LAKMTp4f8g4xGFtfAvVJ09KorMzFgoue1M55Zo9G2eQcKmGN3DNwbaz4vRQMd0nNh5v0+P8893D1f8B8qsi3juiOJ3oReI07CSV2PB5i69lSbA/Xl8P6z9D/rVe4l48TkiyY5AkFbtLIvnLJx0mycXSkc9Ga4YNvomsghw0KeSWY6jsC1cbmIsLFO3g7oLRsCcVlM4mnEWSVTvd2CqOrJ/f2IgptDCoZXQ/1ZW9YTceIzwaI8dCiuzESkl2xISE8F2bzeygO5BKRZ1Z9nQhyrsIrWh9tdlZ1zW1nOALq3J5TEbWXNuiKChUPNvLxk4vkqjzvTi8nZpwBz1+PHrMaPQcDTqWc0yxZ8mxemfXtAlIdjmRcT6HQtr73PFlaaRjgDhMzDHoGgxtpHSvp2fp1O7bw83VfP7k057YWe5w2jQhYSfaHTa0g5kwHjN2NGbsEWZsYWYqOqummoXa60WpAkcGJWYNUmPIEw878So37VJFUDKgCIkx5KHr7W/hygT49QizruLXLOaAG5tczSZihO12DmLiGMQsZPYYMWb1pCPENCRnniyTMcQwim5Utam0rFOruhnVewFrrjhXhsMnLbC3uiBnKM6fX0oxh3ZVSqq8PCP4dJ5qZ952bN3UqPHI0KORcUwzyhGeZHN2cUAf+Vk13bvgs6+FjQlaB7IJiRl5k6g5GyuVDCUZqjut5H6AjEVWdpqKX5jNcwT5nGbPKdjaIvf/fpzNh13ay1TsjaHxiOHRiDkW95PRSra5B9WKrBI2gK4hVCgiflAPmqL8Fw2e5XNrJnTq2SSTeibomsuJGQSleKlXC/oXzmt1ElGPx4gdjZFjAUUh7bLaUiOXyZN2lLMPqTVHSsE+slU+aSTFECydWReOrUfTrOjSemYmtsdUvvrDWo612iUVFunwOiMnbKIHDadJe6MuLzMefcbBsVNLkCHJcqMUtFdIiFXOkZsm08FCI1NTdakFExpsBCmXdbFZA32pc6/1nDWqnXtlt81giA1eyz4ooLis675fLl+tlmHXl0CerUBLqd2YMaDXI0B/ZOw0EsAtTQFOneTUTbAJsRkDEbApUmt4h+5a9VSrHAQlp+Xok6FSVEnmMuh/cz2XvGlxoB+gz+8mWscda1F49UP8EL4n/d1TJQU+6SZNZKfIMxE/X90tBqw2hQyLIggSdgxCeARC+AghTqXQdOQOBV6TNz4l6lS17Fg2QdXiVevJBshEhHWfY1WNa87sE2fSF9kC7dxyNnknNcb4/teyCqV/tRAVemtT4Co8bMUEDejdGNCbEaA3R6BXnWW2u2Sg9FryIfhSctbNJZeiMsEZIB2Tbdxd6yE4CHeFOKATx+7bZbYwyLyrDUUxiXHyLsZvw2JPpl8uiC8c0R3U4M6DusaPAbgdAXB7BPDQbM9a440OCsE6n2IjW8h5akHKHllB24Weuk/VqwTT6OQd/qxlCSq/FvDNku7GIdVBGsSnVT29Z3vUd3viwEDHdIK5P0z+IRaE19OxXYR9fTB++aQj1HRreumJVYQQQHYai7JR+1YRJCh5rwOUnZyf6NkiOyUqHkLD1kbUtVNnl3N3OKN/21XIXcUHmMbgetZruadUCsUjxfEY0aMxcixSB+5FIzQHMiqTLIfbXGvPBeLNG0RrD1mtQ87Q0bZmD7kHsoorIRbXnB6Dka+2Kx5rbLzbaFhC/LQeviEfttlYJatWjccJj8bJsWCdkAmVHlMvkbKBlpbmMGyqRXIKwQ0HZlLyyRWYUqktRibEio7IrrsxhUfh5F/a/XUqg/OaTpZWoYev92ZSCClDKrV7WVw/laDAC43HixmNl2ORHFG5JVNjNn1ImTgGm2NsiC8qRrKUkNnKEQu4sGR979L1RHFXBroOue6otjJkt7MNSvpL4c1hR151yI2BGz0eN3Y0bo4FfViE4uRBA8uJaQT9mrnE3GEU2gbvCnKvQnKEvVHwqgcLtRd9J40/1nEiy6xt+K61+tLqxacTq2Dm/WmrHAgtlseg5IxlwROfdJiSGlJLENzZMSdlUw5yYrWnlE1FmhujlALhyJrJphnXoocpGeOQkHlnT9Lfm7X0XeayL9pvGQ2e/3gtYeUMOzHjkaJHI+VYvEeoqB5RpCYK0rNFkfLFIyvLJZoG8wgwo9YYmkX7wAg2ZDW5wI4tRCSPQcqhcL8KJJIm73FeO4MLjMSOxwePxsexWK+azmQ9JxMahGH2Dr9sgpXUmGRXls5KR2HE2cBKMbioCC3JmSBbFcfgYy3UP8X5Z9t47o21Xi45lnu9sW48OsxodBwL8bIyHkvzKseG7Nfa2uV8JOhoCibQ2WtG7ouQr5M01vLB25KSNVVpyymP4rPW61ez9epVDGvxg9Yy4ZN9lh+PFDsaKcdieyssLaWAdc6ykR0B3CgkWpxzR3pMSZUcOEQLyRIV9ElJXCzYgv/qrtoxSDkhtp9aurKjSHc/mnT3J0l3JYeZ4I5UTspVtilkGEIqvVU4r0aetSuplEomuRA7FYVw7ohzlW2ir9jLfk5bm5OKK8cWY20cjxg9GjHHIjopG+GDgi3WIN9F3qtyRd6VnI7BGRO1nMppPTSORhVbTPOpN+h363Vs7jJi1qqML4/jzLbXnRZkxYGmU1o2OTUeI3Y0Ro75K1c0siyujhAhNKJ0qolTHbrBOkXUQrBcvEqhINJ7SBUlGsWR97CglsZgZOOE1HRbratfh+/1QJQeNPt0sZcBtlK3ir9Xt6tOl6dJdnf5sp97/aqfm4bsAwWOHVFbQxIiZXW5Gc/G2jhUHk2oEUG7VY8smIqHmVCJ5DhJO95z9vPGA+e/X573G5oT3kkPltvrL8Nhjs1zgxuw3y8O2VaYgB4Bd3057oeHTikr10wFuMibXCyQcLEXhGlnnVXIZZuybLw0L2LlAhQfslutndK5Krvj4P05uNOuozSbHQ1Wa9xDL4PZfKG0r2b7AzV+CiY9jwC+uRx8cxj8FDPEdCdI6mCsDh2Jq4KHh9KT/jY+GxChuCgvcbm5bEFBcdqmpLoK4ULwB1cDoJ/d/0bRVtv4op/ESbtvI2a/eTUBr6qpu0tL6vBAhhCQow8pJARiCDmCpiu9alNdt9R1bRpeSdlWQo/JOGMh5irF3imb86uEO+1hk5O1NY3NHZ3faPdhMBGNOLIykUOldROfWh/b0agxY1FzxE5yKiEX32I1pUfN0BK5tVZr46qb49Z0MbZqm320HEwrDcZDUY7QUiY9CjUvrWWRLN08Xs+v7q/by0PMZ5iLG4GTM+qE7tIyoZs2TqScqiGSklNN0VOS4zaZLZWcQ+la1QzJ5wprloMFGplsSrKHKgV9fgFkV9jeMpeVyN4yiq19hG///KcXFPz723VXt7ldBCbjR6NHj0XP0bheZacOIZKwdGxRSJtAlnNZW6Wlih6yRtCH/qsm1U7aIdTnoBJJ/uv+FvTsOFS+cmTS4GIzvqy30Zao1ITR7YYjLoxGDI9FDB8hJoUaajcluBhIm0hyeoAJbEDthead87LBrUhfo6iUqqRVUCDS4n0ljkLMS162g8yhE/5b9jOYD3xYHI0LMxYXR+IKsUmY7JGkKS3XnqQhbWTOhEy4Oc3Uc9dygoMDQg+Tkb1YWWnqvcSRjGTLhW3G9sV26C0ntseB7dvt5tVoxNixiDnWa81SkuMAMTmjXVPGNmlZ27wURripElqOJisTM6wFSpGkI0PNtVr2Veu/qZEMtwGs0YTBQ3fCTYPZTRLYoBHY8GNlxv6UzNgh6Q0BihsEcPPUXfctM14q5OUIoUZqJs2w4aeg4m0IxhWodeh0X5yl8/dCn8TGdq+kb7RfDyXHqoZej0aEGYuIY3rRQCZyq4z8qTcXGfKEW67Sa7BUhYyYZceVduSl53M31lu2QboWtu5aHYEIkey1Xq1vSScnfS92CPYTs9+v3ngejQo7FhVHPJSqyjXuUOzJ2KgAfHbN+Rg0kmBtwYdjE0ODuG/aQIe0CF0CuY+gHxqZUWxi15GNzX6Qy/Ax2MuiJLgRSP5wNx1OzyxeW3bAupp9nPz4gqYfZfeiX5P169dhOBj9ibWUM4dtv30agi7dtGos+a5DV930oRoFKahE9XlJmKyKOpUWuytIfV2PivCPiyovL93YXymxSu1uvgnP3ubtqav5x+d9on8E2l8jr17XEPO7OSAtd7P55Gt55nfb6evKDdndkHpif+L28zOHbb996lNEfE3ewpf3ruXiEuVEQLADtPDqvg8veC/3nhW4mi5XRSkfh9PJ5RikZsd1EX/A9N/sNvvODbeYyF06xk7u5B4juSPiBh/z6vbT+xMcutuHZIwnVlnPHLb99mns0ps0AR6IX+SQVg7Wc6m2qViSk9NB8A2qwGPLya7qE3yEyhZBtcSejiJJxhzooHl1W64fa5t8bSf4qLdD29Le2uQeRp2vfkJy+TXxN7JF42tSZn2yfk1h+bqR812LM8TLNbir263+vd6/EmfzOpzNS5xTiF726lVF8AFUSsm2WpOi50A+JOebVwy9awzSeFkBjSXI/iTmLodPX4XzV2tOYGPihsk3QI6XuE6f1seeSLhPw9HTAd6FYzgGMlxteCXK9nUo25coA9waMGl1diE0VtlZVmxbUi2ZzuSREeamc/b4uhPCYOnwx3AmbBAM1atQ/h/DQYYN6KQ5bJvvmLTGfPfUlHS1835+91O7nbxdBDDJt+NZOO7PsV81+AWmtigVpOeuhnRpNaqkqq9A0sPvIr9zCF46w7eytMKU4yW1JlkgJrnzyNFeTLeyhR3IbtwGN4D57urT7R2yj5czNQwOY7ZIHJ471+zo8/e1VACUAdJBXYQ0X4I0v0S6FVYtGukeikysIx+GUFENUiV7G0NQphkimw05C6VvetGaQ5Y+GtAzyb8K6fXgtnF73LrDEH/xDk76W/Ea75du93h8C3QRvuYSfHf44NyT65QwQeE8MFlN6BZAtu6NdBLVoRYLDQLvHJLuWTXbTEcmUTDVDYV+Ab4vbkADmt8Klk9BbDnNZUpLfnv/AMc9P2XnQlhThS9uNjJ+DSt9KJF4xdBdQ6Y56ZaHEzetIw3rkbwV3R2jlSa4nVUqJXWTfXDei5OWBkrZSscRDikt71Ha74Ih2bfxpbX9OsvFDbmuC0lZTV9mK28sIyezYcrCFz/ePlzdtwmefvupLTssnKD+5Ke/CXwB5Px6yHk35EXOvkpHydizgG9LNrVa3QBzQ3xjUNKg4SA5pBkSu1jlsj+V5L7EbvtrIF9N6RXcZeEotIGnMHbRUBIKog+Z3kZ5b5JbSY+zXcjKMvlS8wkNSDHSZHV/DFA3F6BuX4+63TfRA+foK2S0QI7oFx38dXGZSCGvs5pSVy66zsj2yOJbthNe4BCVTnQJ6uu+eRvyvVXwHZtuVrDfzq7q4HNmgrM9G+f95/xe/YCdmAvahlU0VcMlF3ht2ViWsyvISfANo7JB7gFryNYSrGHY5GwUfE0D+mUv5puLqTuRX4TINRe+Cfw+z/LdnnkuNae07mjW+/8HdzEF+lIK9vj3hGTPulxUGTp59mgwuTmSlTtDklcUImZ4N8nrUHKExmm1K98jt1JLiJdQcGzyLzjY7XXqY1sozl0nX2SsRAD5u14PIbj9ZQ4a/MU08KU07PH5kC6JdOpZFuZcVnLUvjVhopRgaodAzzEXbyoyn2YVWe2yLKFmDYtpI9BwxPPvnvXrrejfbhrKW2mFIKnPLN20lR2EiwkwlxJgdhNAxiObTNrp1o0LzmTnXC8mpmxaDvhC9ZSptYR0Eh7fIzi3mBx7F+GL/H+GHeya5ff3LT1shNnnWf/L3UOdAfV4Mer2UtT3BF0tl95R6d4TQ2d6rTrmc6wxJg6eUpHWB71VyP+aKzL6rmwJsuoG/8SV/zNQ3+9idqC9dw00qgtI8JdGYX8oCgPzmOTeoxQ5SS7TuyNWHELWhVLznZk4MRXrQtW1+RSskRVSQsrZwgkkeLsn0X9Zhd1Nwq4DEgez+tXd3HQx7PpS2PdE3qoaRU/wNsUGlSQASw3GqgLX03TStUTZJdOTGIk2oSSjFJQuKW2jvgD2U9LORTuPQ1d2i28fru2O+mKM+VKM94RVpI62UE65AzIChnAwygZk8nIlGCIpky6eLNJNV7NwkH1S0plRwfko+i85tYE4X4y4vRTxPR7dNyeXfkOlyt2YWk4uAO0ebDCJe23JWMtRyf4izPXmLIJqytI1K7KOgS+Z1V+d6tIPtWx6O90B+d69LV+9iYdkbAj+xGMNrxi6a8gUmFfKVo6tIzt3JduSfaxVW11Ch7T1cra6JFt9N7LlKHM1StaPjQ7Rx2My9mU1bDXT18/s2NVVLcZM8t18fi0LFvfnTnb8rDfRXgCvfj28u4dMC7Ly4BECO/LzzFBLqcqOiCxXgES2kT1Tj65VS3Az4KI1ZVKygZG5Gzof3l0X26/DCxG0rKiv1xkP3hEb3QWY8usx5d2YckV2FzFdVaskwr526UlCzssmeG20KQiDJdhahmkaZaEnBYc8XKnm6+sxXfcUa5juLrrcpIefgPLbJ0Blgr4Fmv5sNPdv/Xz1A3Yi250luUMoq0KtL1Z6lFXB2+jZS0W3KfgDl6B+YpO2kk4uoc5EBM3v0wnO2KrpziXh6ySy8Pru01VZLOGsQf3lql0P/ndjNU1w35lrm2HT9GK1ctiIC8zDBZgfTTlOfcBOzK1nUxDvOsCAL/Yht0bGDXfeNzjimp3NmOnI7ppib5VmDQ/dHOY0BurTAuDZfmJP7fbL8szry1l9SDRGxwdLrAfC2vGhu4ZMY+3OVTjd4joSZK9UL4nZQqMH5G4khakCTV44aeaqkTzrLpuau+zJrEf9rg97V9HS9aeW8WnK5hLacCB1yCww9uh1ZJO/fJT3/QrIklIXQKtfD+3uIVOyBTHNhkS9V7lUWK61gv+V21iD7MqPNjk5iQRXIlfhwu1ylVXh7CEDrXsFtLSaq23+eP88Y8Uj2ElHCBvgXa7zDPCekC3IzwG4dDa4Z3iGUx+wE2gpdCdYek0GHjhoJ3fuNVm/9D5HW8lyrV3p4CMVp6NcHtYRG1sJLCSc5hl2wv0yQ9uev2clZkuoV3Px8K22tOFMy91tv3q4eZqW9omww3fiHX3IwcFT2U+vAaPvcmlYitWX4q2OcjWP68bmEn3q4CfrnozcYRUz4hLio5x8wAN275a8uq0N4Nd2O19Ndz6t5bj7ltS3WlaL7dBynM3GHsmNrA10PR09Fdx5FNz1GLjrI7gHYuIKlaeb1r3J3mzrbJVrEDTyO6co61R15wbfM/Slk+Y1yJat6kE5ugz3XdtT14FfbBVm2XH2VMa7bw9Xd1Uwf7s56dkM/TPpqQR9Gfw8BvxHBk9LjqyCJvbwMw25iNZstYvGe3LDycTIyWWjmkFeImUll8EGQdSQ6qldCP9z+1657Hlzzr9bQn9sg8+wP1j6CcyEggUDdhQGzBgMmCMMWBV9KD4jtnLTriKLUbkYA6VjkXW75qPS1hGmPrMLukKc2+wcksFIOenLGFjO/9V+eTDwbqDg/ZrfOWVTCik3CuR2DMjtEch9NbLTPRMyQsPB5ypdlzqZam22HpInU+FQIia8yYxgXE3P3RlHDX/RcuGkP8PZn1WHWk1+fwETm8syF8Xel486zEpnaaMPU4hJGikCe53IM1Wjc80tFNBlilZV9gC40HVWvbU+dAG0OacTWNlY9OEdN0y8NIcdDunMXIiHVCiMSIoej5Rj4Tk2ROMMa2myuYUJukkWGazvsiEgNwQEhAJHSfnqFZHvwRVjY2mVYENxDFIOBYmlg3oFIXFEQng8Qo4F7NSt3Lydcpe7JMVgsq7a54A4XrtxurYgJ3lgKjmxZ1VMdFHOvrOTa/NGtJJnN7bPTg5QsNEhBXyQGpEPMx4fx8J37AjK0F7d9dCMtOR1HdM/C/oO8Ff4Jo6xQ9RVj+/7gjjvldwSHdm0UbzW7zePGa7ISDPB+R303GO6vv4Cak7r5zfYB9GIfNjx+DgW22W5wnnpYclZjrGppm2ppjbTHL4DXWedj96ZWGyAF+jZye3oTUmb2KLa3zyKvNpjkR6FET9eXPcnxfVQdO6pGw3H1ItcihOD3D5oDbUoW/iM7Yjh0XbvlJLKR/bShdxnV1rL/SxG/LEQcoLXOpkQHpEQPR4hx2K6YSX3quVOVFvGnNdkXe4cki6Wiy2l9U6skXvFbFjI8RrvQ4xRyqkRCHkS3nsC+pD7SsF053bJhdm8/dfP7WG4k3j3AdEhFyYzIkU8HkXHoryShgLFd1NNSYZz9VGWACiknGLWsBwSc3FK+itX6HBpPdcMSW1EzrmPYTMvZMoRP3bEWOyITJjxmDgW33O3NWsL08g5pkbIbI3LKkX8w9y19STLucHlhiwLmXDIFm+PxVXZwjCK9zpRlZzkt1Z6ndyIfNjx+DgW30uLPiBWQ6ODhAJ5wiaSLEPIJbfWJbnK27agm5OLvbs31phUc7KVUot13GgyoiCh02S7DiMUy48/5ODgqZG6d2cuZBEV5Ax2xtcQI177pCAIEfMJkAcE8SAN6mJtpWalfGiVcz69gGK12le02tqPHL8h/41WYAHhTQkPGDsEkk/pIV9vVQ5jnHwzIY/ftFqsv9mPMkCW3yiMwoQZgwlzhAltjKsRWk97aA/PhhLDGGLKsXICSTG4oFyyUOsN0rFIg96mNESLUxh5IRNfvbyKezZ5t6TieyFiKsSobxfcrFqXDs1QFkewFu3QkJJsaJPV0c2BEYnj8QJKXlXTOvVRh+nxcicnQf0RNda1yh31gyWY1JzEhyhrSU3ufIlIyjy4StGn4mziavZ2UN6rRg6Yy/2iaexuOzlleUOeLeah1Yhc6PG40Ee4SCZYubzeN2Lfo0dclwbjyGwL1QI7UnJCvwU50EK+QrBr2EjtucdufORRuHiRVG3RMURzjHx/FiM0IiM8HiNHHjENuihma6NT0Hw2CNityaVHpLNc1WKG/QTkbOsgqHVfgvGyK5fg0bwf0TpOCSZnUaJHpMSMR8mxeDL0qHBy7Fy33kzqMA5lYlS6w3Ag3q2lBimfbbQBKZY3zkbves9Jyo51FEqe86s9Tut1VsIjUmLHo8QeocTZ2jDjVXCpdVki8TFW7YhKlX1ibNhZuU3H1FhqyA0pQY8qZypWazi7/5IxxIzChR/PPPxJ5hGqMtU134er8zqsw8rNOk2HUpFN6WgouNq66UnXznirwS9PzC2nlMO56mOveSybvKT5vN3czyXlekf+W1iEcMK8pOT9ZP45zScpI9WSzGzW0kP5vOrjsGsZV58m05lGUCHHH3Jw8NRbExHHHWv8BZ3F79B7KnNgOaiH/Ldx6AjhLZJ1uhLHlOTkWKOe8Ix+eu5r1Gm3sbK0mdNDkygxDqO2WsJv9V/c6m22lho8t++/mn3ceupAm1EfzSI11m4UxvQYjOkjjCWWDYOmGmfkAvWcTPHZSnfsqpSc9E42axhVqEVp5AEuVsvZWWTDyjaur2Vsbyd/FlT10BxQoEVqO2z4EeLOO4igBir8KFTwGFQcGTyVTfPdyG0KqUiTxkwI770qsKO6cbArTSEhF8gxtFYUgTMvbKimQq7tEirUyxXEd6w+aMjEBQvivoxabIFYU+1ldQVGf7y+HqTjkKytbRFf9KoSPxZGocKMQYU5QgVFCqUrUkDdEMS5r9U66i5ENsrLFn2dkhR/ZXlRmQy70RnpgFx3H7u9zI89uzHZ8r3Thx0/6jcc8yMdRwHdjgG6PQJ6hkPROssxbt+41x589U1BWGRtm5WjxaHEkKXXuFex5SJ35kkf+cyplXCZK9q5HvXCGS3yXKPeb4WRQ27oDbG6gIVXLUud+qjDjGi4e83FDx1riIsNxbaaVSRp7WhBRbfOUQi5c6raUqwQJtG3In7rJDPYPCGhTrwGbIdFHGg4segKX9v1FaxGLtt7++c/vaBp8mdptCw37Y1HFY9H1bHgkStyLFiOoa4sUU5sndGgomcH6Rtq8C6bDqHSYqhks+UeoRZLRnKsdByDqlX+tZ4s7WBq/yriya1ljRo6yxKfpunHKNBfUp8PU0RtbXIP0VLh3pS0Pwu9dI6+QkbCxdWgqobOR6SXk185Ntvl9K0yWtUz9jfSnktIHnfdjTtcU/W4KLK0p0sor26Fo1m7neH156i+noBtHbcVw+ExqNAjUKEPU0E1BIkysZUot612sQHuDmGle4Qg11krlZqGaMxco6upVISaiNQsD72+LqDiqz9s35InfZJhHSydQD9QXHbCPveI+WAMZgwGeAQGDo+diucpRruYLDngjLQ21Jw6W8x1pZVrMBPWFItVtQXlkf76FLLKnbxLFzLwdAEumelE8wJjoumEeOdlIx/eEX8g9f57+u50KuwYVJgRqDCHqSjVleJqrvBNGa/FLD3srPNBd7ERwwgWQbnikhQXZYm3ISfTRmsjyv0CKlaSY/2E07Mp0NouOZjFSWVFGgpZ7MbA3o6AvT2MvZTY5ZhHajlVNshjtXbSJsd7aGsOTXbsVuhy751haX9ZIrybL1DqAa7oIuwXV9zy/psRzAdZD5xN6IxLXSa0yKH86xl41crg5QuDYep90U4O9fmcrO/JNO+1Tjo0i8c2xyHBAILq3dhKpBTMIECxh+bldqSzdymexMnG1XobJ6/p6eD126+3OPh6/60uxGE8avRo1ByJ2JZsgUPKtfhKKfQOqlTqgXXL+EfbQta35qrcIqKKLp2UTjnHLA0KKI9AzVe/fc6SNO+zl281f9D8/TM3J+S2W6kt/bg4scZxPKJ4NKKOBfbQoi6xOt2oNugP5WxSNnbIqkQdRAXIemlvgLjDgW1KXi40thGaBBFlDKJ2GNFyQ8NTanWCANlHz2TFj1Hj8WNG4+dItA9IdRFl8AdHKsJ+QjPOaViWXOIKnjwhLUPYQZDpyUP7cy6eqk+y3FjT38THbdoRHvt4PV8Gnt0eThzcxrkr+DVD49FhR6PjSAKQo9bRmpbgsJoqxvaave8t2RS7VuS117Yj+OjAzelsMSC73JihWEoPI9Dx0q0N11VJteU6Xd3s82s7GNkbcowegxo/WjbgT8kGLASgTRDoZBzkR05EKYVgCmyls0km1ML4rqfkkvOFa62+tuRJQcE3c245ZQc1L5TiplBcVCLpjMTY8HhE6NGIOKbWobwZnglxI9reIsFEemK5fi87jbjhA3R7MPhOb8YGJ5kbKAAHzrbiRiDi5dmqNZWy6yT0RtchAvwD+mY89Hk09I8E9EwAv5JmqsnAHdFwU4pJOQDbbnvoUXvTNbXGcFhNW82+gAEQZpuji9Ff3aBr9ibFv45bafGZhRMwY8djxozGzJFQzqlmKJLg4J6CVQX/lz1xJYZkDVmD3Mr1HJhILkN03JiQaVEwcl+lMXUEZnZdwy6W8Y6Gmu9wScXuitbh4olZE/DrF0hYq888SPW60dujpuw7PL3cYphhBYCxuAp9nmt0LsSmkw1W9yIX3CjrQwmZZAdKN4m1Bzkbl1bsRdjt7ciFcbO72+HQuQRl+eK36eerOvkFaRK8/T0Cr4TbSZikPofn+WO6lv/1dn09MTeL7jxrb3o3Tw/Spcc9XSi0rknEIvxuBhDdzHnR4XWjt0dNfaNSZSsPtLhsEk1Ft+5ZerBq4xGPi0qKnS612YgIrFXnDMnR2OWaXDyNAdpaFpTbxB5keXtY25DK+F3vS4gXG0yEinz3l8kv7erT57nM/KlSavLTp9n7SR5OQgm0V7PPA3lrragQzWEEw/il1KBhv4IJlyHPFyH/ctRUGsyVEKxP3rpaUi1Zlr41JIQcv7Fe+t76kvENW7t2JpApqqgKt+qacpcjv7a3Co7qXhYmJuVxPmA38DGfFPjGtiJBbs4ZmiluCO27BWXymI3GdjLb16T24b5y+mCR9jVjd7ajQ4qjspXFh0LS9696B4mgqQZWxoAIjl6WKEgXuTPOlqig67xNJemi3GaX4V3L3+c3onuUKIzI8dB+vhruzdpqPrfd78iq8zHd78df/4Sd+CIWNkKYlKs4KyRvNzoZrhEIBg03g7neOShILk3a5AB/TtloU4kaTKDvxfdou7+X/YXPa2E59Fd9u9zeIfnOCm66HG6+GO493RVJKWkGwi6RcnDkOfheA7Gj2LNcJ2KNUxSthUspRsHTE6JqU7oEE8rWjSFnwf3iyqf1Jqyblz39x2ObLRY5j92oRVZfDra5GOw9voOQPjnoU2mWY4vLlrJNSbnEXUsDu1wae+NdldsjUyollp6jM5nkLLjLl87tkzoKn7p7yfL0pl7fUDh2IBKyJNKTY5U+4cuB07PHHTl7WqvcBxdSkYs+XC5w055cNUgNHQFPZaUcEKNS3Iht8wb5YQxG+jLKXN5Ou3//29/9fuLMh+EGwoM9in6/7Ph093D1afAMDw3Cs343WXQ0hAe+/jJJeM+n2xlwhC/+E7kp8orpRId/X9ytssj7hvPcV7cIiHV5cltaDy9vqJUsxJpTcDfqdbjvHndwwBQzOmnKzFA0cMzFkTRF86XIIr3tyhqI0YYg6JWuJDvyuEnlrBBVqqWdhjvpl7ifBDs+BSbubPIzfE3K18vLzPLDcK2kzHPJsYeC2JC2COZ3j3PB/Ofltyb42QDevhZ4fiXwfAx47kjtQqq9M5mh/QDEZNDdWOgdVq7Wihy7a6NtU6mbrocWH9I0MOsdh61HA/5p6+nVc4L4z18A7O0TDUtzuIH/mi0bGOze0y3Qu7Ogx4++liLvuXN+Y9wR6E2ztssNK1QKe9uTET3fFeZ+yrXE2j0ceI4+snF4oyqUg5E+8lHHTEegX3yS86Dvm8fedYTQl//kpDXptSV7496vAD8USf1rMedXYn5suucObZPJ1uZsQ34Sgsmmsg/aVt9qLi5FE3xQch+FidWCGNdiCkkqZMH8DTBP2+cTN1A3briuXa+WUoZn7MQ6nIJ1eJ1LD2d69DDVIXtEROh0AFdUaIGdCyEhSUm9mO5zZyPZoZKbbQIbcjESZjh8DFIaOtGvnIHz0nfMbq/u7xEkB79+D88N17GMm4va1Re4nb8sqcDc/tPim0/TfdOpxFeCzq8D/dj6bPLOtx5jqyalFLtzKsLNhJyQoaveUjH4l5HINAvtyTHJilM38C/Vmvg3B12axmMSI06Wz3JDGTLJn5pI+od3yy2L/77CefuQ+yKI/urN/wf66TU+"},"review_C":{"sha256":"35856b07b7f07b9d775128c3d7f29c35c46a9858e3f6c489625a58c61f1bdbaa","bytes":4904,"file":"Reviewer_C_3_Cases_COMPLETED.csv","compressed_base64":"eNrtWNuO2zYQffdXDPYl3q3W0V22i6AI2gQJEKRBs0VQIIBBkZRNWBJdkrKz6PbfOyQlXzbOIpcC7UOwD2uRnOHMmeE5lBTfCr5bCBZQuZLKLNb8NjBKkNqObZQsa97Yn4pTqdiiEu2Sq40SrQkYMURzExiilvhPy05RvkCbZmMCqcRStOhGdmbTGbTfEKE4OzzbjbmyvlddQ9oFOlsvdEcp17of4h82Cp+EbBdbUu9XagyQGhsL+vcT6I3xFrdvpeEBExr34AvFiZZtQDHMRds1JVcjUktWN1E2ny91M13PjSKinWdZXNz5qUW1SeK7ZS1LUgfD6uBzFt9fE8yKpIyqghSzMuLTYkoiRrIqjSkri3SWslk0DeM0TVlB2SwJy6yMi6qc8mnI8yhJqPcY5JMwuHhHasMVVLyuoUKIwaw4cLFcmRVsamIqqRqQFWjZcNCUVJWsGRYLSMu81YZoA78QBApI5XyRurYrUmg4PusJvKwAvfo9pEJTIIwJg/iTGjdUnEMjFQcjGq5xgBhgfIMRlLyy4ythjPVoY1sq2bUscPvb5yFIDQRX8i1v61vQG0I5A+wJEC3ubfMJYGcd75PaEd2HLdufLoKLG+vMNyYabWW9xVgGePqUPomQjQaB0HbRw1i84NZB69E4h8RHIIyjuLe+/DQgE7h5GAxcINEhumlEy32Qj/TeYj4aRRO4urqRBiPBRjcE297Fz9urq/k+AxgfCn4JP8A+OBgfpWLtLuEJRPmQ+Ci27t8MgNmgMAPr+RU38H4MK3hv83M5+Yrhk9lxxIrKVnPaGbE9SnGCNfNQ20rK1hlWojoqy2Q0el/KD5z9lf19Eajtgi7CMAlC93fx1hCDSGfHdRzv119iE2jDCbPd76kIchinjesqHItybAWHlk3Wt8b02JfQEB9VhMldG8AUrnH0CeSXE3gtsUzCrBAhQQEPBiU17dAAMQTce+PLiE1pWYj9CFhn7CmzEopdD0QHJdKVusWVpUCOsLaB66tS89acuKyIqLWDSfE/O2TNxq5wJxIhbjY1VioAR3yAtMaRDTG7Yw/jEMN+1jINO4wb9mAdY/X85eunrxZPX7999+y3OeRYppZzphfHAeYjhhzaRMkpXRZRmN65mbskCaM7rHslVHMnN7xd1Nh2iyK7S7IwDnrz4Ous71sFBY8oi2dplCVJxkpOaMyrIsmmIYnTIkkJDUmY5MiwPJuFvIrDCimYRDzJS0byWc+oUWgp9SkY1dE1AoedS5StDTTkg2g6x6QYdRaGsF5qx6vdZlMLjsTwB0cIFSO3gSuR92Fbu5YEq+8Rj9DSgt4TxM6eEzdo3WHYK8cD2AHCnLEFqlzL71aCrnrj/MRWuv1PjckScfIushA0oetTDzi4FrVcKtIMfl7IHabc3kJJ0Ldb5ljw/kqH0Q6FxQg8QnY/y9f79Ada9lisMCTSHzlKLH+Y21NAJ/COg+02XLXvW88o3mznecWeaBwbsAdSY9ew233CSOa6K7E/qLFY7Fn/3t64Cd4VvMysbMYdZuSEzKaFVOb9TRy1/nwSzmkgrqKH0l4NFX1sGRS5yj9ZBn3Iiy/uocpXfW3Ry/8AR9evX45kv1MPKCrMZ+M5h3OI4vNXgDo/OjxXHyvJjepa6iYbwa4tcwMm++jLXT/qrw2WjTun8NIqeJ/TYD+2yfiUjlNBav61EUYP3sZ5Px70h3YcDwXGIWR/PNZ2i30Vxul+1t3u7D0YrwJboR3329nH9gzbTd3Jxh1/d5J0VoNsuzlYWJ+P1xAmMTa8TjtVN8LghfqM/pxIIfYXvkMMFx7Saqt7qEU+zlbe0xyMrrZ3HBRge89HkTsrQlFafJsMJd8kQx9bYSGLlE6nGV7ts5xRwmhZxYTEYZbTspxlRZ7zuKAlTmSsivN0GqU0pCGL0iznYf5dhr7L0HcZOoeje+X6d7Xoi1D9T8To2zTIKY/TsugBxXnu3ylkT/EDmsOBHwQrCfqcL+17zkF70vB6vfTHczzuK38NfvnlQW1wozdnZQZUV2PQ9n0V3XVCuwM+iElf44MMHURFW3pDgRDtfjEKmdh6rzjRELVGk9C9Hx0+GHl5wj4J+in7eQmGz0uu1/y4/5hkfzbkU/ozHf0DxRp8vg=="},"review_gate":{"sha256":"0497d9380e6aefb9aff7db826c7bad36557912f40bffe0f9395303c0cde65af5","bytes":428,"file":"adjudication_gate.json","compressed_base64":"eNplj8tOxDAMRffzFVbXgBheEuxAiBXzDZancduIvEicogHx77jTChh1m3PPzfXXBqCRKOSwc9T3bJoH2N7en03vmUfLH/iIYxQua/C0ApT3VjKJjeGXXS9KiW5kgy2VE6WGFbo8gsBsCv5vPOVD9RRwmdJGnxwLK5VceS7QAfsY36626I3zGk1kM97doGRLbmpqEgdjQ98cha46h1paS5ttEuwy8yerR+ag4U6dv+bJflEBEiXOMGeBjLHTVnLuAJnfq9XrQAaG5Sfw0bA7D3rRyLB7ft3BvAvaOMQsQMFAZ7UAipAo1OYyXDSb7x+yPo8e"}}

def decode_source(k):
    s=EMBEDDED[k]
    b=zlib.decompress(base64.b64decode(s["compressed_base64"]))
    assert len(b)==s["bytes"], f"Byte length mismatch: {k}"
    assert hashlib.sha256(b).hexdigest()==s["sha256"], f"SHA-256 mismatch: {k}"
    return b

def table(k):
    return pd.read_csv(io.BytesIO(decode_source(k)), keep_default_na=True)
raw=table("source_trials")
audit=table("multirubric")
rec=table("reconciled")
A=table("review_A")
B=table("review_B")
C=table("review_C")
gate=json.loads(decode_source("review_gate"))
print("SHA-256 validated:",list(EMBEDDED))
print("Shapes: source", raw.shape,"multi-rubric",audit.shape,"reconciled",rec.shape, "A/B/C",A.shape,B.shape,C.shape)

In [ ]:
# Hard guards against partial input, mismatched cohorts, duplicate model-qualified trial IDs, or unfinished reviews.
KEY=['cohort_key','trial_id']
for name, frame in [('source',raw),('audit',audit),('reconciled',rec)]:
    assert len(frame)==710, (name, len(frame))
    assert not frame.duplicated(KEY).any(), f'Collision in {name} model-qualified IDs'
    assert not frame[KEY].isna().any().any(), f'Incomplete IDs in {name}'
source_set=set(map(tuple,raw[KEY].to_numpy()))
assert source_set==set(map(tuple,audit[KEY].to_numpy()))==set(map(tuple,rec[KEY].to_numpy()))
assert rec.cohort_key.nunique()==7
assert rec.flagged_for_human_review.sum()==159
assert rec.reported_task_success.sum()==74
assert len(A)==len(B)==159 and len(C)==3
assert gate['human_review_complete'] is True and gate['unresolved_cases']==0
assert gate['notebook21_mdlm_repair_64_trials']=='pending'
assert rec.loc[rec.flagged_for_human_review.eq(1),'human_adjudicated_task_success'].notna().all()
assert rec.loc[rec.flagged_for_human_review.eq(0),'human_adjudicated_task_success'].isna().all()
assert set(A.review_id)==set(B.review_id)==set(rec.loc[rec.flagged_for_human_review.eq(1),'review_id'])
assert set(C.review_id).issubset(set(A.review_id))
assert A.record_fingerprint.notna().all() and B.record_fingerprint.notna().all()
print('PASS: 710 unique model-qualified trials; 159 fully reviewed; zero missing-arbitration cases.')

In [ ]:
# Verify that reviewer votes and fingerprinted source outputs refer to the exact cases being analyzed.
reference=audit.set_index('review_id')
for person, votes in [('A',A),('B',B),('C',C)]:
    assert not votes.review_id.duplicated().any()
    for k in ['cohort_key','trial_id','record_fingerprint']:
        lookup=reference.loc[votes.review_id,k].astype(str).to_numpy()
        actual=votes[k].astype(str).to_numpy()
        assert np.array_equal(lookup,actual), f'Unexpected {person} {k} mismatch'
    assert votes.reviewer_id.notna().all() and votes.evidence_note.fillna('').str.strip().ne('').all()
    assert votes['human_task_success'].isin([0,1]).all()
    assert votes['human_strict_final_valid'].isin([0,1]).all()
    assert votes['human_expression_valid'].astype(str).isin(['0','1','uncertain']).all()

# Reviewers A and B agreed on task and strict-format decisions for every case.
for col in ['human_task_success','human_strict_final_valid']:
    assert A.set_index('review_id')[col].equals(B.set_index('review_id').loc[A.review_id,col].set_axis(A.review_id))

# All 3 uncertain expression decisions are resolved by Reviewer C.
aunc=A.loc[A['human_expression_valid'].astype(str)=='uncertain','review_id'].tolist()
bunc=B.loc[B['human_expression_valid'].astype(str)=='uncertain','review_id'].tolist()
assert set(aunc)==set(bunc)==set(C.review_id) and len(aunc)==3
assert C['human_expression_valid'].isin([0,1]).all()
print('PASS: original-output fingerprints, reviewer agreement, and third-reviewer arbitration.')

## Part II — Frozen post-review outcome ledger
The decision rule is specified before statistical analysis: 159 reviewed tasks use the adjudicated task outcome; 551 other tasks preserve the original automated task outcome. A complete human-review gate applies only to the **159-case selection**, not the full 710 records. Separate alternative rubrics remain *sensitivity measures*, not substituted ground truth.

In [ ]:
# Join additional verifier fields and original compute metadata without discarding their provenance.
merged=raw.merge(rec[KEY+['review_id','flagged_for_human_review','human_adjudicated_task_success','state']], on=KEY, how='left', validate='one_to_one')
extra=['explicit_final_numeric_correct','strict_final_line_correct','expression_witness_found',
       'automated_evidence_based_success','automated_format_and_evidence_success','record_fingerprint']
merged=merged.merge(audit[KEY+extra],on=KEY,how='left',validate='one_to_one')
assert len(merged)==710 and merged.flagged_for_human_review.notna().all()
merged['reconciled_success']=np.where(merged.flagged_for_human_review.eq(1),
    merged.human_adjudicated_task_success, merged.task_success_reported).astype(int)
merged['outcome_provenance']=np.where(merged.flagged_for_human_review.eq(1),'human_adjudicated','original_automated')
assert merged.task_success_reported.sum()==74
assert merged.reconciled_success.sum()==72
assert int((merged['reconciled_success']!=merged['task_success_reported']).sum())==2
assert merged.loc[merged.flagged_for_human_review.eq(1),'reconciled_success'].sum()==43
assert merged.outcome_provenance.value_counts().to_dict()=={'original_automated':551,'human_adjudicated':159}
print(merged.groupby(['outcome_provenance']).agg(trials=('trial_id','size'),task_successes=('reconciled_success','sum')).to_string())

In [ ]:
# Explicitly separate held-out primary problems from secondary trajectories.
secondary = merged.role.eq('secondary_new_trajectory_same_problem')
main=merged.loc[~secondary].copy()
secondary_df=merged.loc[secondary].copy()
assert len(main)==680 and len(secondary_df)==30
assert main.reconciled_success.sum()==67 and secondary_df.reconciled_success.sum()==5
assert (secondary_df.cohort_key=='dream13').all()
print('MAIN:',len(main),'trials,',main.reconciled_success.sum(),'reconciled successes')
print('SECONDARY:',len(secondary_df),'trials,',secondary_df.reconciled_success.sum(),'reconciled successes')
print('Original-to-reconciled flips:')
flips=merged.loc[merged.reconciled_success.ne(merged.task_success_reported), KEY+['problem_id','action','role','dataset','task_success_reported','reconciled_success','review_id']]
print(flips.to_string(index=False))
assert len(flips)==2 and (flips.cohort_key=='dream13').all()
flips.to_csv(OUT/'table00_human_corrections.csv',index=False)

## Part III — Publication tables with problem-level denominators
Rates based on trials can describe decoder outcomes, but **independent failed problems** are the unit for confirmatory claims. We report exact binomial confidence intervals for descriptive problem-any success and paired tests on shared problem IDs; these intervals assume the sampled problems meaningfully represent a population, an assumption that cannot be established by these small curated cohorts.

In [ ]:
def exact_ci(x,n,alpha=.05):
    x=int(x); n=int(n)
    if n<=0: return (np.nan,np.nan)
    lo=0 if x==0 else float(beta.ppf(alpha/2,x,n-x+1))
    hi=1 if x==n else float(beta.ppf(1-alpha/2,x+1,n-x))
    return lo,hi

def make_surface(df):
    items=[]
    for (key,act),g in df.groupby(['cohort_key','action'],sort=True):
        any_by_problem=g.groupby('problem_id').reconciled_success.max()
        n=int(len(g)); t=int(g.reconciled_success.sum()); p=int(len(any_by_problem)); ps=int(any_by_problem.sum())
        cl,cu=exact_ci(ps,p)
        items.append({'cohort_key':key,'cohort':g.cohort.iloc[0], 'role':'+'.join(sorted(g.role.unique())),
            'model':g.model.iloc[0], 'decoder_policy':g.decoder_policy.iloc[0], 'action':act,'trials':n,
            'reconciled_trial_successes':t, 'trial_rate_descriptive':t/n,
            'distinct_failed_problems':p,'problem_any_successes':ps, 'problem_any_rate':ps/p,
            'problem_CI95_low':cl,'problem_CI95_high':cu,
            'human_reviewed_trials':int(g.flagged_for_human_review.sum()),
            'automated_only_trials':int((~g.flagged_for_human_review.eq(1)).sum()),
            'mean_preservation':pd.to_numeric(g.token_preservation,errors='coerce').mean(),
            'mean_native_forward_calls':pd.to_numeric(g.forward_calls_native,errors='coerce').mean(),
            'median_runtime_seconds':pd.to_numeric(g.wall_seconds,errors='coerce').median()})
    return pd.DataFrame(items)

surface=make_surface(main)
surface.to_csv(OUT/'table01_main_680_action_surface.csv',index=False)
make_surface(secondary_df).to_csv(OUT/'table02_secondary_30_separate.csv',index=False)
print('Main cohort-action rows:',len(surface),'distinct cohorts:',surface.cohort_key.nunique())
display(surface[['cohort_key','action','trials','reconciled_trial_successes','distinct_failed_problems','problem_any_successes','human_reviewed_trials']])

In [ ]:
# Model-native primary comparisons: actions paired by the same failure ID within each cohort.
# A group only enters paired analysis if global and treatment evaluate the identical problem set.
from scipy.stats import binomtest
comparisons=[]
for (key,role),g in main.groupby(['cohort_key','role']):
    if 'global' not in set(g.action): continue
    global_g=g[g.action=='global']
    by_prob_global=global_g.groupby('problem_id').reconciled_success.agg(['max','mean'])
    for action in sorted(set(g.action)-{'global'}):
        s=g[g.action==action].groupby('problem_id').reconciled_success.agg(['max','mean'])
        assert set(s.index)==set(by_prob_global.index), f'{key}/{action} has unpaired problems'
        paired=by_prob_global.join(s,lsuffix='_global',rsuffix='_selective')
        a=paired['max_selective'].to_numpy(dtype=int)
        b=paired['max_global'].to_numpy(dtype=int)
        wins=int(((a==1)&(b==0)).sum()); losses=int(((a==0)&(b==1)).sum())
        discordant=wins+losses
        pval=float(binomtest(wins,discordant,0.5).pvalue) if discordant else 1.0
        diffs=paired['mean_selective'].to_numpy()-paired['mean_global'].to_numpy()
        # Within each problem average over repeated seeds, then resample whole problems.
        rng=np.random.default_rng(RNG_SEED+len(comparisons))
        sampled=np.mean(diffs[rng.integers(0,len(diffs),size=(4000,len(diffs)))],axis=1)
        lo,hi=np.percentile(sampled,[2.5,97.5])
        comparisons.append({'cohort_key':key,'role':role,'action':action,'n_problems':len(paired),
            'selective_problem_any':int(a.sum()),'global_problem_any':int(b.sum()),
            'selective_only_problems':wins,'global_only_problems':losses,'discordant_problems':discordant,
            'problem_any_rate_diff':float(np.mean(a)-np.mean(b)),
            'mean_per_problem_seed_rate_diff':float(diffs.mean()),
            'cluster_bootstrap_rate_diff_low':float(lo),'cluster_bootstrap_rate_diff_high':float(hi),
            'paired_exact_two_sided_p':pval,
            'inference_note':'no discordant problems; p=1 is not evidence of equivalence' if discordant==0 else 'exact paired problem-any sign test; exploratory'})
paired=pd.DataFrame(comparisons)
from statsmodels.stats.multitest import multipletests
if len(paired):
    paired['holm_adjusted_p']=multipletests(paired.paired_exact_two_sided_p,method='holm')[1]
paired.to_csv(OUT/'table03_problem_level_paired_tests.csv',index=False)
print('Paired comparisons:',len(paired));display(paired[['cohort_key','role','action','n_problems','selective_only_problems','global_only_problems','mean_per_problem_seed_rate_diff','paired_exact_two_sided_p','holm_adjusted_p']])

### Confirmatory sensitivity: Dream independent primary cohort
The main held-out Dream cohort contains **8 distinct failed problems × 3 actions × 5 seeds = 120 trials**. The other 30 Dream trial records are repeated trajectories on previously used SVAMP problems and are secondary evidence.

In [ ]:
dream = main[(main.cohort_key=='dream13')&(main.role=='primary_unseen_problem')].copy()
assert len(dream)==120 and dream.problem_id.nunique()==8
check=dream.groupby('action').agg(n=('trial_id','size'),success=('reconciled_success','sum'),problems=('problem_id','nunique')).reset_index()
dream_any=dream.groupby(['action','problem_id']).reconciled_success.max().groupby('action').sum()
check['any_problem_successes']=check.action.map(dream_any)
print(check.to_string(index=False))
assert dict(zip(check.action,check.success))=={'global':2,'open_last_25':0,'open_last_75':5}
assert dream_any['global']==dream_any['open_last_75']==1
row=paired[(paired.cohort_key=='dream13')&(paired.role=='primary_unseen_problem')&(paired.action=='open_last_75')].iloc[0]
assert row.selective_only_problems==row.global_only_problems==0 and row.paired_exact_two_sided_p==1.0
print('Held-out action contrast (last75 - global) = +7.5 percentage points per trial, but 0 additional distinct problems.')
check.to_csv(OUT/'table04_dream_independent_heldout.csv',index=False)

## Part IV — Fragility, persistence, and compute trade-offs
The next analyses are **descriptive**, not a pooled model leaderboard. A preserved incorrect output can indicate answer persistence, while same-last-number flags are only a weak proxy for repeated semantic error. Compute metrics can be compared **within** a specific decoder and model, not indiscriminately across models.

In [ ]:
# Leave-one-problem-out (LOPO) stability for every same-problem intervention contrast.
lopo=[]
for (key,role,act),p in paired.set_index(['cohort_key','role','action']).iterrows():
    g=main[(main.cohort_key==key)&(main.role==role)]
    by=g[g.action.isin(['global',act])].groupby(['problem_id','action']).reconciled_success.mean().unstack()
    all_diff=(by[act]-by['global']).mean()
    for problem_id in by.index:
        sub=by.drop(index=problem_id)
        diff=float((sub[act]-sub['global']).mean()) if len(sub) else np.nan
        lopo.append({'cohort_key':key,'role':role,'action':act,'omitted_problem_id':problem_id,
                     'full_per_problem_mean_rate_difference':float(all_diff),'leave_one_out_rate_difference':diff,
                     'absolute_influence':abs(float(diff-all_diff))})
lopo=pd.DataFrame(lopo)
lopo.to_csv(OUT/'table05_leave_one_problem_out.csv',index=False)
print('LOPO rows',len(lopo))
print(lopo.groupby(['cohort_key','action']).absolute_influence.max().sort_values(ascending=False).head(10).to_string())

In [ ]:
# Wrong answer recurrence and exact output persistence (whitespace normalised).
def canon_text(s):
    return ' '.join(str(s).split()) if pd.notna(s) else None
persistence=main.copy()
persistence['exact_output_repeat']=[int(canon_text(a)==canon_text(b)) if pd.notna(a) and pd.notna(b) else np.nan for a,b in zip(persistence.original_output,persistence.repaired_output)]
rows=[]
for (key,action),g in persistence.groupby(['cohort_key','action']):
    same=pd.to_numeric(g.same_wrong_numeric_reported,errors='coerce')
    exact=pd.to_numeric(g.exact_output_repeat,errors='coerce')
    rows.append({'cohort_key':key,'action':action,'n_trials':len(g),
                 'n_with_both_texts':int(exact.notna().sum()),'exact_repeat_n':int(exact.fillna(0).sum()),
                 'exact_repeat_rate_when_observable':float(exact.mean()) if exact.notna().any() else np.nan,
                 'reported_same_last_number_n':int(same.fillna(0).sum()),
                 'reported_same_last_number_rate':float(same.mean()) if same.notna().any() else np.nan,
                 'mean_preservation':float(g.token_preservation.mean()),
                 'reconciled_success_rate':float(g.reconciled_success.mean())})
persist=pd.DataFrame(rows)
persist.to_csv(OUT/'table06_exact_recurrence_and_proxy.csv',index=False)
print(persist[persist.cohort_key.isin(['mdlm18','bd3lm23'])].to_string(index=False))
print('Missing original texts are excluded from exact-text denominators; never treated as non-repeats.')

In [ ]:
# Within-cohort cost and preservation summaries (no cross-model forward-pass pooling).
cost=main.groupby(['cohort_key','action']).agg(n=('trial_id','size'),
    mean_wall_s=('wall_seconds','mean'),median_wall_s=('wall_seconds','median'),
    mean_preserved=('token_preservation','mean'),
    mean_native_forward=('forward_calls_native','mean'),n_forward_logged=('forward_calls_native','count'),
    successes=('reconciled_success','sum')).reset_index()
cost['success_rate']=cost.successes/cost.n
cost['forward_pass_unit_note']='decoder-specific native forward calls; not globally comparable'
cost.to_csv(OUT/'table07_within_cohort_compute_and_preservation.csv',index=False)
print(cost.loc[cost.cohort_key.isin(['bd3lm23','alodlm15'])].to_string(index=False))

## Part V — Vector figures for the paper
The figures are derived from reviewer-reconciled success definitions. They do not reproduce pre-adjudication plots or imply a significant intervention effect.

In [ ]:
plt.rcParams.update({'figure.dpi':135,'savefig.dpi':200,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
labels=['Global regeneration','Reopen last 25%','Reopen last 75%']
acts=['global','open_last_25','open_last_75']
vals=[dream.loc[dream.action==a,'reconciled_success'].mean()*100 for a in acts]
fig,ax=plt.subplots(figsize=(7.2,4.3))
rect=ax.bar(labels,vals,edgecolor='#2D4157',color=['#799bb4','#a5b5c2','#375c7a'],width=.62)
for bar,a in zip(rect,acts):
    subset=dream[dream.action==a]
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.65,
            f"{int(subset.reconciled_success.sum())}/{len(subset)}",ha='center',fontsize=10)
ax.set_ylim(0,19);ax.set_ylabel('Reviewer-reconciled success per trial (%)')
ax.set_title('Dream: held-out primary repair trials (8 problems × 5 seeds)')
ax.tick_params(axis='x',labelrotation=0)
fig.tight_layout()
for ext in ['png','pdf'] : fig.savefig(FIG/f'fig01_dream_heldout_trial_success.{ext}',bbox_inches='tight')
plt.show();plt.close(fig)

In [ ]:
# Same trial data, but count DISTINCT successful problems. This prevents pseudoreplication.
values=[int(dream[dream.action==a].groupby('problem_id').reconciled_success.max().sum()) for a in acts]
fig,ax=plt.subplots(figsize=(7.2,4.3))
rect=ax.bar(labels,values,color=['#799bb4','#a5b5c2','#375c7a'],edgecolor='#2D4157',width=.62)
for b,v in zip(rect,values):ax.text(b.get_x()+b.get_width()/2,v+.045,f'{v}/8',ha='center')
ax.set_ylim(0,2.1);ax.set_ylabel('Distinct failed problems solved (of 8)')
ax.set_title('Dream: held-out problem coverage after adjudication')
fig.tight_layout()
for ext in ['png','pdf']:fig.savefig(FIG/f'fig02_dream_heldout_problem_coverage.{ext}',bbox_inches='tight')
plt.show();plt.close(fig)

In [ ]:
# Pre-specified cluster bootstrap intervals for selected contrasts; exploratory.
wanted=[('dream13','primary_unseen_problem','open_last_75'),('dream12','exploratory','open_last_75'),
       ('alodlm15','model-native','open_last_75'),('bd3lm23','model-native','tail75')]
plotrows=[]
for key,role,act in wanted:
    sel=paired[(paired.cohort_key==key)&(paired.role==role)&(paired.action==act)]
    if len(sel):
        v=sel.iloc[0]
        plotrows.append((f'{key} / {act}',100*v.mean_per_problem_seed_rate_diff,
                         100*v.cluster_bootstrap_rate_diff_low,100*v.cluster_bootstrap_rate_diff_high))
fig,ax=plt.subplots(figsize=(8.4,4.3))
y=np.arange(len(plotrows))
x=np.array([q[1] for q in plotrows]); ll=np.array([q[2] for q in plotrows]); uu=np.array([q[3] for q in plotrows])
ax.errorbar(x,y,xerr=[x-ll,uu-x],fmt='o',capsize=4,color='#365b75',ecolor='#7596af')
ax.axvline(0,color='black',alpha=.55,ls='--',lw=1)
ax.set_yticks(y,labels=[q[0] for q in plotrows]);ax.invert_yaxis()
ax.set_xlabel('Per-problem, seed-averaged success difference vs global (percentage points)')
ax.set_title('Exploratory problem-cluster bootstrap: 95% percentile intervals')
fig.tight_layout()
for ext in ['png','pdf']:fig.savefig(FIG/f'fig03_selected_problem_cluster_intervals.{ext}',bbox_inches='tight')
plt.show();plt.close(fig)

In [ ]:
# Trade-off inside the same BD3LM23 suffix-regeneration protocol.
g=cost[cost.cohort_key=='bd3lm23'].sort_values('mean_preserved').copy()
fig,ax=plt.subplots(figsize=(7.1,4.4))
ax.scatter(100*g.mean_preserved,100*g.success_rate,s=[150]*len(g),color='#426a85')
for _,r in g.iterrows():
    ax.annotate(f"{r.action} ({int(r.successes)}/{int(r.n)})",(100*r.mean_preserved,100*r.success_rate),
                xytext=(5,8),textcoords='offset points',fontsize=9)
ax.set_xlim(10,102);ax.set_ylim(-.9,10)
ax.set_xlabel('Mean retained-token fraction (%)')
ax.set_ylabel('Task success per trial (%)')
ax.set_title('BD3LM23: prefix preservation versus repair success')
fig.tight_layout()
for ext in ['png','pdf']:fig.savefig(FIG/f'fig04_bd3lm23_preservation_success.{ext}',bbox_inches='tight')
plt.show();plt.close(fig)

## Part VI — Manuscript evidence ledger and reproducibility export
Notebook 27 does **not** assert that a paper is ready to submit. Its claims are linked to concrete trial subsets, denominators, and reviewer provenance. No untested GPU experiments are inserted. Text below is suitable for adaptation into Results/Limitations, not publication without author review.

In [ ]:
# Additional domain stratification highlights differences between math-expression and word-problem grading.
by_dataset=main.groupby(['cohort_key','dataset','action']).agg(n=('trial_id','size'),successes=('reconciled_success','sum'),n_distinct_problems=('problem_id','nunique'),human_reviewed=('flagged_for_human_review','sum')).reset_index()
by_dataset['trial_rate']=by_dataset.successes/by_dataset.n
by_dataset.to_csv(OUT/'table08_dataset_stratified_main.csv',index=False)
print(by_dataset[(by_dataset.cohort_key=='dream13')][['dataset','action','n','successes','n_distinct_problems']].to_string(index=False))

In [ ]:
# Outcome sensitivity: keep task grading separate from formatting and arithmetic witnesses.
rubrics=[]
metric_col={'original_reported':'task_success_reported',
            'partially_human_reconciled':'reconciled_success',
            'explicit_final_numeric':'explicit_final_numeric_correct',
            'automated_evidence_witness':'automated_evidence_based_success',
            'strict_format_plus_evidence':'automated_format_and_evidence_success'}
for (key,act),g in main.groupby(['cohort_key','action']):
    for rubric,column in metric_col.items():
        rubrics.append({'cohort_key':key,'action':act,'rubric':rubric,'trials':len(g),
                        'passes':int(pd.to_numeric(g[column],errors='coerce').fillna(0).sum()),
                        'rate':float(pd.to_numeric(g[column],errors='coerce').fillna(0).mean())})
rubric_table=pd.DataFrame(rubrics)
rubric_table.to_csv(OUT/'table09_scoring_rubric_sensitivity.csv',index=False)
print(rubric_table[(rubric_table.cohort_key=='dream13')&(rubric_table.action=='open_last_75')].to_string(index=False))

In [ ]:
# Pre-registered-style claim audit: classify each paper statement by whether the frozen evidence supports it.
claim_ledger=[
 {'claim':'710 repair trials across seven noninterchangeable experimental cohorts','supported':True,'scope':'observational record inventory'},
 {'claim':'All 710 repaired outputs received human adjudication','supported':False,'scope':'159 adjudicated, 551 retain original automated label'},
 {'claim':'Original 74 task successes reduce to 72 after flagged-case review','supported':True,'scope':'full 710 ledger; partial human reconciliation'},
 {'claim':'Held-out Dream last75 yielded 5/40 successful trials; global yielded 2/40','supported':True,'scope':'Dream13 primary only, reviewer-reconciled'},
 {'claim':'Dream selective repair solved more distinct held-out problems than global','supported':False,'scope':'both succeeded on the same one of eight problems'},
 {'claim':'A universal or statistically confirmed advantage of selective repair','supported':False,'scope':'problem-level comparisons and multiple-testing safeguards'},
 {'claim':'BD3LM23 selective suffix repairs outperformed global regeneration','supported':False,'scope':'0/48 selective, 1/16 global, exploratory small cohort'},
 {'claim':'MDLM Notebook 21 constant-per-block repair was completed','supported':False,'scope':'not included; generation stage alone available'},
 {'claim':'Model-native forward calls provide comparable compute across all architectures','supported':False,'scope':'different model sizes, implementations, counters, and attention costs'},
 {'claim':'All review cases are administratively resolved','supported':True,'scope':'A/B agreed 156; C arbitrated 3; independence procedure not independently verifiable'},
]
claims=pd.DataFrame(claim_ledger)
claims.to_csv(OUT/'table10_claim_evidence_audit.csv',index=False)
print(claims[['claim','supported']].to_string(index=False))

In [ ]:
# Data invariants for export and a paper-friendly, explicitly scoped results paragraph.
assert int(surface.reconciled_trial_successes.sum())==67
assert int(merged.reconciled_success.sum())==72
assert int(surface.trials.sum())==680
assert int(main.flagged_for_human_review.sum()+secondary_df.flagged_for_human_review.sum())==159
assert ((paired.holm_adjusted_p>=0)&(paired.holm_adjusted_p<=1)).all()

results_paragraph=(
    'Across seven model-specific repair cohorts, 710 existing trials were audited. '
    'Two of 74 originally reported task successes were overturned after independent-vote files '
    'and third-review arbitration for 159 flagged cases, yielding 72 partially human-reconciled successes. '
    'The remaining 551 trials retain their original automated labels. '
    'The main comparison comprises 680 trials and 67 successes; 30 Dream secondary trajectory trials '
    'with five successes are reported separately. In the Dream held-out primary cohort, reopening the '
    'last 75% of the response achieved 5/40 trial successes compared with 2/40 for global regeneration, '
    'but both approaches solved the same single problem among eight distinct problems. '
    'The problem-level paired comparison has no discordant successful problems (two-sided exact p=1.0), '
    'which does not establish equivalence. In the BD3LM controlled-denoising suffix-regeneration cohort, '
    'the three selective conditions yielded 0/48 successes and global regeneration 1/16. '
    'These outcomes are exploratory and do not establish a general advantage for selective repair.'
)
limits_paragraph=(
    'The failure sets are small, curated, and not interchangeable across architectures. '
    'Denoising budgets, model scale, precision, and native repair semantics differ. '
    'Repair attempts share problems across seeds, so trial-level counts are descriptive; '
    'problem-level paired tests and cluster resampling are used for uncertainty. '
    'The 159 flagged records received adjudicated votes, but the remaining 551 keep automated outcomes; '
    'reviewer independence cannot be inferred from files alone. '
    'Some Countdown graders rely on expression discovery rather than final-line compliance. '
    'Notebook 21 MDLM controlled-denoising repairs remain unavailable and are excluded. '
    'Post-hoc best-action and oracle estimates should not be interpreted as prospective routing performance.'
)
(OUT/'MANUSCRIPT_RESULTS_AND_LIMITATIONS.md').write_text(chr(10).join(['# Reviewer-reconciled Results (draft)', '', results_paragraph, '', '# Limitations', '', limits_paragraph, '']), encoding='utf-8')
print(results_paragraph)

In [ ]:
# Source provenance, output ledger, and self-verifying ZIP. No hidden dependency on uploaded files.
columns=['cohort_key','cohort','model','precision','native_intervention','decoder_policy','role','trial_id','problem_id',
         'dataset','action','target','task_success_reported','reconciled_success','outcome_provenance',
         'flagged_for_human_review','human_adjudicated_task_success','explicit_final_numeric_correct',
         'automated_evidence_based_success','automated_format_and_evidence_success','token_preservation',
         'forward_calls_native','wall_seconds','original_output','repaired_output','record_fingerprint']
merged[columns].to_csv(OUT/'FROZEN_710_Reconciled_Trial_Ledger.csv',index=False)
A.to_csv(OUT/'REVIEW_A_159_frozen.csv',index=False)
B.to_csv(OUT/'REVIEW_B_159_frozen.csv',index=False)
C.to_csv(OUT/'REVIEW_C_3_frozen.csv',index=False)

publication_gate={
 'n_trials':710,'main_trial_count':680,'secondary_trajectory_count':30,
 'reviewed_cases':159,'remaining_automated_labels':551,'original_successes':74,'reconciled_successes':72,
 'main_successes':67,'secondary_successes':5,
 'primary_dream_heldout_last75_trial_successes':5,'primary_dream_heldout_global_trial_successes':2,
 'primary_dream_distinct_problems':8,'primary_dream_last75_any_success_problems':1,
 'primary_dream_global_any_success_problems':1,
 'mdlmb21_repair_stage':'NOT INCLUDED',
 'human_review_complete_for_flagged_cases':True,
 'entire_710_human_reviewed':False,
 'paper_ready_without_outstanding_experiment':False,
 'source_sha256':{k:{'file':v['file'],'sha256':v['sha256']} for k,v in EMBEDDED.items()}
}
(OUT/'NB27_EVIDENCE_GATE.json').write_text(json.dumps(publication_gate,indent=2)+chr(10),encoding='utf-8')

all_files=sorted(p for p in OUT.rglob('*') if p.is_file() and p.name!='SHA256_MANIFEST.json')
manifest={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in all_files}
(OUT/'SHA256_MANIFEST.json').write_text(json.dumps(manifest,indent=2)+chr(10),encoding='utf-8')

zpath=Path.cwd()/'RepairScopeX_NB27_PostAdjudication_results.zip'
with zipfile.ZipFile(zpath,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=8) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,arcname=str(p.relative_to(OUT)))
with zipfile.ZipFile(zpath) as z:
    assert z.testzip() is None
    stored=json.loads(z.read('SHA256_MANIFEST.json'))
    assert all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in stored.items())
print('All exported files passed post-ZIP SHA-256 verification.')
print('ZIP:',zpath.resolve(),'size:',round(zpath.stat().st_size/1024),'KiB')
print('Total files in ZIP:',len(stored)+1)

### Interpretation and next work
This is a **recomputed observational synthesis**, not new benchmark generalization. The strongest statement is about *failure persistence and protocol dependence*, with no supported claim of general selective superiority. Complete **Notebook 21's MDLM 64 repair trials** and rerun a version of this notebook that explicitly incorporates and audits them before a full experiment freeze.

For Colab, the final code cell provides optional ZIP downloading; otherwise use the Files panel on the left.

In [ ]:
# Optional, Colab only. Keep this cell disabled unless you want the browser download prompt.
AUTO_DOWNLOAD = False
if AUTO_DOWNLOAD:
    try:
        from google.colab import files
        files.download(str(zpath))
    except ImportError:
        print('Colab download helper unavailable; retrieve the ZIP from:', zpath.resolve())
else:
    print('Results saved at:', zpath.resolve())
    print('Enable AUTO_DOWNLOAD=True in this final cell for a Colab download prompt.')